# TopCoW: 3D Bounding Box Detection and Multi-Class Segmentation Pipeline
**Circle of Willis — ROI Prediction and Medical Image Segmentation from CTA/MRA Scans**

### Project Overview
This notebook implements an end-to-end deep learning pipeline for analyzing 3D Magnetic Resonance Angiography (MRA) and Computed Tomography Angiography (CTA) scans. The primary goal is to isolate and segment the **Circle of Willis (CoW)**, a critical arterial polygon in the brain. 

The pipeline is divided into two major phases:
1. **Phase 1: Region of Interest (ROI) Localization:** A 3D Convolutional Neural Network (`ROIRegressor3D`) predicts a tight 3D bounding box around the Circle of Willis to crop out irrelevant background volume.
2. **Phase 2: Multi-Class 3D Segmentation:** A custom `UNet3D` architecture takes the localized ROI and performs multi-class voxel-level segmentation to identify specific blood vessels and structures.

In [ ]:
import numpy as np 
import pandas as pd 

# Input data files are available in the read-only "../input/" directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
import torch
print(torch.cuda.device_count())

In [ ]:
!nvidia-smi

## 1. Environment Setup & Dependencies
In this section, we import the required scientific computing and deep learning libraries. 
* **Data Processing:** `numpy`, `pandas`, `nibabel` (for parsing `.nii` medical images)
* **Modeling:** `PyTorch` (for 3D CNNs, Custom Datasets, and DataLoaders)
* **Image Processing:** `scipy.ndimage` (trilinear interpolation), `skimage` (Frangi vessel enhancement)
* **Visualization:** `matplotlib`

In [ ]:
# ============================================================
# IMPORTS & SETUP
# ============================================================
import os
import re
import random
import warnings
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import zoom
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')

## 2. Global Configurations & Path Definitions
Here, we configure the hardware accelerator (GPU/CPU) and define the root directories for our dataset. 

We also define crucial hyperparameters that govern the **ROI Regressor** phase:
* `PATCH_SIZE`: The standard 3D volumetric input size (96x96x64) forced by resizing.
* `MARGIN`: Contextual padding added around the ground truth ROI before cropping.
* `BATCH_SIZE`, `NUM_EPOCHS`, `LR`: Standard training parameters.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# PATHS
# ============================================================
ROOT       = "/kaggle/input/datasets/anushakhare03/dataset"
OUTPUT_DIR = "/kaggle/working/predicted_roi"
CHECKPOINT = "/kaggle/input/datasets/anushakhare03/results/roi_regressor_best .pth"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── sub-folder helpers ───────────────────────────────────────
def unwrap_single_subdir(path):
    """If a folder contains exactly one subfolder, return that subfolder."""
    try:
        entries = [e for e in os.listdir(path) if not e.startswith(".")]
    except FileNotFoundError:
        return path
    if len(entries) == 1:
        sub = os.path.join(path, entries[0])
        if os.path.isdir(sub):
            return sub
    return path

ROI_LABELS = unwrap_single_subdir(os.path.join(ROOT, "roi_loc_labelsTr"))
SEG_LABELS = unwrap_single_subdir(os.path.join(ROOT, "cow_seg_labelsTr"))
IMAGES_VAL = unwrap_single_subdir(os.path.join(ROOT, "imagesVal"))

In [ ]:
# ============================================================
# HYPERPARAMETERS
# ============================================================
PATCH_SIZE  = np.array([96, 96, 64])   # fixed model input (after resize)
MARGIN      = 20                        # voxels added around ROI before resize
BATCH_SIZE  = 2
NUM_EPOCHS  = 50
LR          = 1e-3

## 3. Data Ingestion & Indexing
Medical datasets often consist of uncoupled files (images, segmentation masks, ROI coordinates, and metadata). 

We utilize regular expressions (`re`) to parse file names and securely map each patient's MRA/CTA scan to its corresponding Ground Truth Region of Interest (`roi_loc_labelsTr`) and Segmentation Mask (`cow_seg_labelsTr`). Finally, we calculate the exact number of paired training and validation samples available.

In [ ]:

# ============================================================
# DATA INDEX
# ============================================================
img_pat = re.compile(r"^topcow_(ct|mr)_(\d+)_0000\.nii(?:\.gz)?$")
roi_pat = re.compile(r"^topcow_(ct|mr)_(\d+)\.txt$")
yml_pat = re.compile(r"^topcow_(ct|mr)_(\d+)\.yml$")
seg_pat = re.compile(r"^topcow_(ct|mr)_(\d+)\.nii(?:\.gz)?$")

def build_index(root_dir, pattern):
    out = {}
    for r, _, files in os.walk(root_dir):
        for f in files:
            m = pattern.match(f)
            if m:
                out[(m.group(1), m.group(2))] = os.path.join(r, f)
    return out

image_index = build_index(ROOT,       img_pat)
roi_index   = build_index(ROI_LABELS, roi_pat)
yml_index   = build_index(ROOT,       yml_pat)
seg_index   = build_index(SEG_LABELS, seg_pat)

paired_data = []
for (mod, cid), img_path in image_index.items():
    roi_p = roi_index.get((mod, cid))
    seg_p = seg_index.get((mod, cid))
    yml_p = yml_index.get((mod, cid))
    if roi_p and seg_p and yml_p:
        paired_data.append(dict(id=cid, modality=mod,
                                img=img_path, roi=roi_p,
                                seg=seg_p,   yml=yml_p))

val_images = []
for r, _, files in os.walk(IMAGES_VAL):
    for f in files:
        if img_pat.match(f):
            val_images.append(os.path.join(r, f))

print(f"Paired training samples : {len(paired_data)}")
print(f"Validation images       : {len(val_images)}")


In [ ]:
# ============================================================
# ROI PARSER
# ============================================================
def parse_roi_txt(path):
    """
    Returns float32 array [x_min, y_min, z_min, size_x, size_y, size_z]
    in voxel coordinates of the original volume.
    """
    size, loc = None, None
    with open(path) as f:
        for line in f:
            ll   = line.strip().lower()
            nums = [int(x) for x in re.findall(r'-?\d+', ll)]
            if 'size' in ll and len(nums) >= 3:
                size = nums[:3]
            elif any(k in ll for k in ('loc', 'origin', 'pos')) and len(nums) >= 3:
                loc = nums[:3]
    if size is None or loc is None:
        return None
    return np.array(loc + size, dtype=np.float32)

In [ ]:
# ============================================================
# COLLECT SAMPLES
# ============================================================
def collect_all_samples():
    samples = []
    for fname in os.listdir(ROOT):
        if not fname.endswith('.nii'):
            continue
        parts = fname.replace('.nii', '').split('_')
        if len(parts) < 4:
            continue
        modality = parts[1].lower()
        pat_id   = parts[2]
        if modality not in ('ct', 'mr'):
            continue

        img_path = os.path.join(ROOT, fname)
        roi_path = os.path.join(ROI_LABELS, f'topcow_{modality}_{pat_id}.txt')
        if not os.path.exists(roi_path):
            continue
        roi = parse_roi_txt(roi_path)
        if roi is None:
            continue

        samples.append(dict(img_path=img_path, roi=roi,
                            pat_id=pat_id, modality=modality))
    print(f'Collected {len(samples)} samples')
    return samples

samples = collect_all_samples()


## 4. Volumetric Preprocessing & Normalization
Before feeding 3D volumes into our neural network, we must account for the distinct physical properties of CT and MR imaging:
* **CT Scans:** Hounsfield Units are clipped between `-100.0` and `500.0` (optimal for soft tissue and contrast) and Min-Max normalized.
* **MR Scans:** Voxel intensities vary widely, so we normalize based on the 1st and 99th percentiles of non-zero (brain) voxels to eliminate outlier artifacts.

### 4.1. Tight Crop and Resize (Target Localization)
To optimize memory and model performance, we dynamically crop the original volume around the Ground Truth ROI (plus a margin). Since ROIs vary in physical size, we use **trilinear interpolation** (`scipy.ndimage.zoom`) to standardise the crops to our fixed `PATCH_SIZE` while mathematically projecting the bounding box coordinates into this new localized space.

In [ ]:
# ============================================================
# INTENSITY NORMALISATION
# ============================================================
def normalize_ct(vol):
    lo, hi = -100.0, 500.0
    return (np.clip(vol, lo, hi) - lo) / (hi - lo)

def normalize_mr(vol):
    brain = vol[vol > 0]
    if len(brain) == 0:
        return vol.astype(np.float32)
    p1, p99 = np.percentile(brain, 1), np.percentile(brain, 99)
    return ((np.clip(vol, p1, p99) - p1) / (p99 - p1 + 1e-8)).astype(np.float32)

def load_volume(img_path, modality):
    vol  = nib.load(img_path).get_fdata(dtype=np.float32)
    orig = np.array(vol.shape[:3], dtype=np.float32)
    vol  = normalize_ct(vol) if modality == 'ct' else normalize_mr(vol)
    return vol, orig

In [ ]:
# ============================================================
# CORE CROP FUNCTION 
# ============================================================
def tight_crop_and_resize(vol, roi, patch_size=PATCH_SIZE, margin=MARGIN):
    """
    Strategy
    --------
    1. Build a tight bounding box = ROI expanded by `margin` voxels on each side.
    2. Clamp to volume boundaries  (no padding needed, no overflow possible).
    3. Crop that region out of the volume.
    4. Resize the crop to `patch_size` using trilinear zoom.
    5. Rescale the ROI coordinates to match the resized crop.

    Returns
    -------
    crop_resized  : float32 ndarray  shape = patch_size
    roi_in_patch  : float32 ndarray  [x,y,z, dx,dy,dz]  in resized-patch voxels
    start         : int ndarray  crop start in original volume (needed for inversion)
    zoom_factors  : float ndarray  resize factors (needed for inversion)
    """
    vol_shape  = np.array(vol.shape[:3])
    patch_size = np.array(patch_size)

    # ── 1. tight box with margin ─────────────────────────────
    box_min = np.floor(roi[:3] - margin).astype(int)
    box_max = np.ceil( roi[:3] + roi[3:] + margin).astype(int)

    # ── 2. clamp to volume ───────────────────────────────────
    start = np.clip(box_min, 0, vol_shape - 1)
    end   = np.clip(box_max, 1, vol_shape)

    # Guarantee at least 1 voxel in every dim
    end = np.maximum(end, start + 1)

    # ── 3. crop ──────────────────────────────────────────────
    crop = vol[start[0]:end[0], start[1]:end[1], start[2]:end[2]]

    # ── 4. resize to fixed patch_size ────────────────────────
    crop_shape   = np.array(crop.shape[:3], dtype=float)
    zoom_factors = patch_size / crop_shape
    crop_resized = zoom(crop, zoom_factors, order=1)   # trilinear

    # ── 5. rescale ROI into resized-patch space ───────────────
    #   original ROI loc  →  shift to crop origin  →  scale by zoom
    roi_loc_in_crop  = roi[:3] - start          # shift
    roi_loc_in_patch = roi_loc_in_crop  * zoom_factors
    roi_size_in_patch= roi[3:]          * zoom_factors

    roi_in_patch = np.concatenate([roi_loc_in_patch,
                                   roi_size_in_patch]).astype(np.float32)

    return crop_resized.astype(np.float32), roi_in_patch, start, zoom_factors

def invert_roi_to_original(pred_roi_in_patch, start, zoom_factors):
    """
    Converts a predicted ROI (in resized-patch voxels) back to
    the original volume voxel space.
    """
    loc_in_crop  = pred_roi_in_patch[:3] / zoom_factors
    size_in_orig = pred_roi_in_patch[3:] / zoom_factors
    loc_in_orig  = loc_in_crop + start
    return np.concatenate([loc_in_orig, size_in_orig]).astype(np.float32)


## 5. Dataset Definition & DataLoaders (ROI Phase)
We encapsulate our preprocessing logic inside a custom PyTorch `Dataset` (`CoWROIDataset`). 
* **Inputs:** Localized 3D Volume crops `(1, 96, 96, 64)`.
* **Targets:** Bounding box coordinates normalized to `[0, 1]` relative to the patch size `(x, y, z, width, height, depth)`.

We split the available data (85% Train / 15% Val) and wrap them in PyTorch `DataLoaders` with multi-processing workers for efficient GPU pipelining.

In [ ]:
# ============================================================
# DATASET
# ============================================================
class CoWROIDataset(Dataset):
    def __init__(self, samples, patch_size=PATCH_SIZE, margin=MARGIN):
        self.samples    = samples
        self.patch_size = np.array(patch_size)
        self.margin     = margin

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        vol, _ = load_volume(s['img_path'], s['modality'])

        crop, roi_patch, _, _ = tight_crop_and_resize(
            vol, s['roi'], self.patch_size, self.margin
        )

        # Normalize ROI to [0, 1] relative to patch
        roi_norm = np.concatenate([
            roi_patch[:3] / self.patch_size,
            roi_patch[3:] / self.patch_size
        ]).astype(np.float32)

        # Clamp to [0,1] — safety net for float rounding
        roi_norm = np.clip(roi_norm, 0.0, 1.0)

        return (
            torch.from_numpy(crop).unsqueeze(0),   # (1, 96, 96, 64)
            torch.from_numpy(roi_norm)              # (6,)
        )


# ── train / val split ────────────────────────────────────────
train_samples, val_samples = train_test_split(
    samples, test_size=0.15, random_state=42
)
print(f"Train: {len(train_samples)}   Val: {len(val_samples)}")

train_ds = CoWROIDataset(train_samples)
val_ds   = CoWROIDataset(val_samples)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)}   Val batches: {len(val_loader)}")



## 6. Phase 1: 3D ROI Regressor Architecture
We define a 3D Convolutional Network (`ROIRegressor3D`) to predict the exact 6 degrees of freedom of the bounding box. 

* **Encoder:** A series of 3D Convolutions, Batch Normalizations, and Max Pooling layers that downsample the volumetric spatial features into deep representations.
* **Global Average Pooling (GAP):** Flattens the 3D feature maps to prevent overfitting and drastically reduce parameter count.
* **Regression Head:** A Multi-Layer Perceptron (MLP) mapping features to 6 output values. We use a Sigmoid activation to ensure predictions remain strictly within the normalized `[0, 1]` range.

In [ ]:
# ============================================================
# MODEL
# ============================================================

class ConvBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class ROIRegressor3D(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock3D(1,   16), nn.MaxPool3d(2),   # 48×48×32
            ConvBlock3D(16,  32), nn.MaxPool3d(2),   # 24×24×16
            ConvBlock3D(32,  64), ConvBlock3D(64,  64),  nn.MaxPool3d(2),  # 12×12×8
            ConvBlock3D(64, 128), ConvBlock3D(128, 128), nn.MaxPool3d(2),  #  6× 6×4
        )
        self.gap  = nn.AdaptiveAvgPool3d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(256,  64), nn.ReLU(inplace=True),
            nn.Linear(64,    6),
        )

    def forward(self, x):
        out  = self.head(self.gap(self.encoder(x)))
        loc  = torch.sigmoid(out[:, :3])
        size = torch.sigmoid(out[:, 3:])
        return torch.cat([loc, size], dim=1)

model = ROIRegressor3D().to(DEVICE)
print(f"Model ready on {DEVICE}")

## 7. Loss Formulation & Optimization strategy
Bounding box regression in 3D space requires a highly specific loss function. We utilize a **Combined Loss** composed of three weighted penalties:
1. **3D Intersection over Union (IoU) Loss ($\beta$):** Directly optimizes the volumetric overlap between predicted and true boxes.
2. **Centroid MSE Loss ($\gamma$):** Penalizes predictions whose geometric centers drift too far from the true center.
3. **Smooth L1 Loss ($\alpha$):** Robust coordinate regression penalty less sensitive to outliers than pure MSE.

We optimize this loss using `Adam` paired with a **Cosine Annealing Learning Rate Scheduler** to guarantee smooth convergence.

In [ ]:
# ============================================================
# LOSS
# ============================================================
def iou_3d_loss(pred, target):
    p_min  = pred[:, :3]
    p_size = torch.clamp(pred[:, 3:], min=1e-4)
    p_max  = p_min + p_size

    t_min  = target[:, :3]
    t_size = torch.clamp(target[:, 3:], min=1e-4)
    t_max  = t_min + t_size

    i_min  = torch.max(p_min, t_min)
    i_max  = torch.min(p_max, t_max)
    inter  = torch.clamp(i_max - i_min, min=0)
    i_vol  = inter[:, 0] * inter[:, 1] * inter[:, 2]
    p_vol  = p_size[:, 0] * p_size[:, 1] * p_size[:, 2]
    t_vol  = t_size[:, 0] * t_size[:, 1] * t_size[:, 2]
    iou    = i_vol / (p_vol + t_vol - i_vol + 1e-8)
    return (1.0 - iou).mean()

def centroid_loss(pred, target):
    p_c = pred[:, :3]   + 0.5 * pred[:, 3:]
    t_c = target[:, :3] + 0.5 * target[:, 3:]
    return nn.functional.mse_loss(p_c, t_c)

def combined_loss(pred, target, alpha=0.5, beta=0.4, gamma=0.1):
    return (alpha * nn.functional.smooth_l1_loss(pred, target)
          + beta  * iou_3d_loss(pred, target)
          + gamma * centroid_loss(pred, target))


In [ ]:
# ============================================================
# OPTIMISER + SCHEDULER
# ============================================================
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-5
)


## 8. Training the ROI Regressor
Here we execute the training loop over `NUM_EPOCHS`. 
At the end of each epoch, we validate the model on unseen data. If the validation loss improves, we save the model weights. To prevent overfitting, we implement an **Early Stopping** mechanism triggering if no validation improvement is seen after 15 epochs.

In [ ]:
# ============================================================
# TRAINING LOOP
# ============================================================
train_losses, val_losses = [], []
best_val  = float('inf')
patience  = 0
PATIENCE  = 15

print(f"\nTraining on {DEVICE} — {NUM_EPOCHS} epochs")
print("─" * 60)

for epoch in range(1, NUM_EPOCHS + 1):

    # ── train ─────────────────────────────────────────────
    model.train()
    t_loss = 0.0
    for vols, rois in train_loader:
        vols, rois = vols.to(DEVICE), rois.to(DEVICE)
        optimizer.zero_grad()
        loss = combined_loss(model(vols), rois)
        loss.backward()
        optimizer.step()
        t_loss += loss.item()
    t_loss /= len(train_loader)

    # ── validate ──────────────────────────────────────────
    model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for vols, rois in val_loader:
            vols, rois = vols.to(DEVICE), rois.to(DEVICE)
            v_loss += combined_loss(model(vols), rois).item()
    v_loss /= len(val_loader)

    scheduler.step()
    train_losses.append(t_loss)
    val_losses.append(v_loss)

    if v_loss < best_val:
        best_val = v_loss
        patience = 0
        torch.save(model.state_dict(), CHECKPOINT)
        tag = "saved ✓"
    else:
        patience += 1
        tag = f"patience {patience}/{PATIENCE}"

    if epoch % 5 == 0 or epoch == 1:
        lr_now = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS}  "
              f"train={t_loss:.4f}  val={v_loss:.4f}  "
              f"lr={lr_now:.1e}  {tag}")

    if patience >= PATIENCE:
        print(f"\nEarly stop at epoch {epoch}")
        break

print(f"\nDone. Best val loss: {best_val:.4f}")


# ============================================================
# TRAINING CURVES
# ============================================================
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, label='Train', lw=2)
ax.plot(val_losses,   label='Val',   lw=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('ROI Regressor — Training Curves')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150)
plt.show()



## 9. ROI Regressor Evaluation
We evaluate the best performing checkpoint on the validation set. To interpret our model's accuracy in a human-readable way, we compute two key metrics:
1. **Per-Axis Mean Absolute Error (MAE):** Calculates exactly how many *voxels* our predictions deviate from the truth across $x, y, z$, and sizes.
2. **3D IoU Score:** Volumetric overlap ratio (closer to 1.0 is better).

In [ ]:
# ============================================================
# VALIDATION EVALUATION
# ============================================================
model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE))
model.eval()
print(f"\nLoaded checkpoint: {CHECKPOINT}")

preds_norm   = []
targets_norm = []
# Store inversion info per val sample
inversion_cache = []   # list of (start, zoom_factors)

with torch.no_grad():
    for s in val_samples:
        vol, orig = load_volume(s['img_path'], s['modality'])

        crop, roi_patch, start, zoom_factors = tight_crop_and_resize(
            vol, s['roi'], PATCH_SIZE, MARGIN
        )

        t = torch.from_numpy(crop).unsqueeze(0).unsqueeze(0).to(DEVICE)
        pred_norm = model(t).cpu().numpy().squeeze()

        gt_norm = np.concatenate([
            roi_patch[:3] / PATCH_SIZE,
            roi_patch[3:] / PATCH_SIZE
        ]).clip(0, 1)

        preds_norm.append(pred_norm)
        targets_norm.append(gt_norm)
        inversion_cache.append((start, zoom_factors))

preds_norm   = np.array(preds_norm)
targets_norm = np.array(targets_norm)


In [ ]:
# ── per-axis MAE ─────────────────────────────────────────────
labels6 = ['x_min', 'y_min', 'z_min', 'size_x', 'size_y', 'size_z']
print("\nMAE in patch voxels:")
for i, name in enumerate(labels6):
    mae = np.mean(np.abs(preds_norm[:, i] - targets_norm[:, i])) * PATCH_SIZE[i % 3]
    print(f"  {name:8s}: {mae:.2f} vox")

In [ ]:
# ── 3D IoU ───────────────────────────────────────────────────
def iou3d_np(p, t):
    p_min, p_max = p[:3], p[:3] + p[3:]
    t_min, t_max = t[:3], t[:3] + t[3:]
    inter = np.maximum(np.minimum(p_max, t_max) - np.maximum(p_min, t_min), 0)
    i_vol = np.prod(inter)
    return i_vol / (np.prod(p[3:]) + np.prod(t[3:]) - i_vol + 1e-8)

ious = [iou3d_np(preds_norm[i], targets_norm[i]) for i in range(len(val_samples))]
print(f"\n3D IoU — mean={np.mean(ious):.4f}  std={np.std(ious):.4f}"
      f"  min={np.min(ious):.4f}  max={np.max(ious):.4f}")

## 10. Visualizing Bounding Box Predictions
Let's visually verify our model's competency. We plot the middle Z-slice of validation volumes, overlaying the **Ground Truth Bounding Box (Green)** and the **Predicted Bounding Box (Red Dashed)**.

# Test Data

In [ ]:
# ============================================================
# VISUALISATION 
# ============================================================
num_viz = min(4, len(val_samples))
idxs    = random.sample(range(len(val_samples)), num_viz)

fig, axes = plt.subplots(1, num_viz, figsize=(5 * num_viz, 5))
if num_viz == 1:
    axes = [axes]

for ax, idx in zip(axes, idxs):
    s = val_samples[idx]

    # reload this sample's volume (was the bug before)
    vol, _ = load_volume(s['img_path'], s['modality'])
    crop, roi_patch, _, _ = tight_crop_and_resize(vol, s['roi'], PATCH_SIZE, MARGIN)

    z_mid = crop.shape[2] // 2
    ax.imshow(crop[:, :, z_mid], cmap='gray', origin='lower')

    gt   = targets_norm[idx]
    pred = preds_norm[idx]

    def box_params(r):
        return (r[0] * PATCH_SIZE[0], r[1] * PATCH_SIZE[1],
                r[3] * PATCH_SIZE[0], r[4] * PATCH_SIZE[1])

    gx, gy, gw, gh = box_params(gt)
    px, py, pw, ph = box_params(pred)

    ax.add_patch(mpatches.Rectangle(
        (gx, gy), gw, gh, edgecolor='lime', facecolor='none', lw=2))
    ax.add_patch(mpatches.Rectangle(
        (px, py), pw, ph, edgecolor='red',  facecolor='none', lw=2, linestyle='--'))

    ax.set_title(f"{s['pat_id']} ({s['modality']})\nIoU={ious[idx]:.3f}", fontsize=10)
    ax.legend([
        mpatches.Patch(edgecolor='lime', facecolor='none', label='GT'),
        mpatches.Patch(edgecolor='red',  facecolor='none',
                       linestyle='--',  label='Pred')
    ], ['GT', 'Pred'], fontsize=8)
    ax.axis('off')

plt.suptitle('GT (green) vs Predicted ROI (red dashed) — in patch space', fontsize=13)
plt.tight_layout()
plt.savefig('/kaggle/working/val_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:

# ============================================================
# TEST INFERENCE
# ============================================================
def collect_test_samples():
    out = []
    for fname in sorted(os.listdir(IMAGES_VAL)):
        if not (fname.endswith('.nii') or fname.endswith('.nii.gz')):
            continue
        parts = fname.replace('.nii.gz', '').replace('.nii', '').split('_')
        if len(parts) < 4:
            continue
        out.append(dict(
            img_path = os.path.join(IMAGES_VAL, fname),
            pat_id   = parts[2],
            modality = parts[1],
        ))
    return out

test_samples = collect_test_samples()
print(f"\nFound {len(test_samples)} test samples")

model.eval()
results = []

with torch.no_grad():
    for s in test_samples:
        vol, orig = load_volume(s['img_path'], s['modality'])

        # For test images we have no GT ROI, so we use the full volume center
        # as a dummy ROI to define the crop region — we use a large box
        # covering the expected CoW region (upper ~40% of the brain volume)
        dummy_roi = np.array([
            orig[0] * 0.2,  orig[1] * 0.2,  orig[2] * 0.5,   # loc
            orig[0] * 0.6,  orig[1] * 0.6,  orig[2] * 0.4    # size
        ], dtype=np.float32)

        crop, _, start, zoom_factors = tight_crop_and_resize(
            vol, dummy_roi, PATCH_SIZE, MARGIN
        )

        t = torch.from_numpy(crop).unsqueeze(0).unsqueeze(0).to(DEVICE)
        pred_norm = model(t).cpu().numpy().squeeze()   # [0,1] in patch space

        # pred_norm → patch voxels → original voxels
        pred_patch = np.concatenate([
            pred_norm[:3] * PATCH_SIZE,
            pred_norm[3:] * PATCH_SIZE
        ])
        roi_orig = invert_roi_to_original(pred_patch, start, zoom_factors)

        loc  = np.maximum(roi_orig[:3], 0)
        size = np.maximum(roi_orig[3:], 1)

        out_path = os.path.join(
            OUTPUT_DIR,
            f"topcow_{s['modality']}_{s['pat_id']}_predicted.txt"
        )
        with open(out_path, 'w') as f:
            f.write('--- ROI Meta Data ---\n')
            f.write(f"Size (Voxels): {size[0]:.1f} {size[1]:.1f} {size[2]:.1f}\n")
            f.write(f"Location (Voxels): {loc[0]:.1f} {loc[1]:.1f} {loc[2]:.1f}\n")

        results.append(dict(pat_id=s['pat_id'], modality=s['modality'],
                            loc=loc, size=size))
        print(f"{s['modality']}_{s['pat_id']}  "
              f"loc={np.round(loc,1)}  size={np.round(size,1)}")

print(f"\n {len(results)} predictions saved to {OUTPUT_DIR}")

## Phase 2: Multi-Class 3D Segmentation of the Circle of Willis

In this phase, we transition from Region of Interest (ROI) localization to dense, voxel-level multi-class segmentation. The goal is to accurately classify each voxel within the localized ROI into one of 16 distinct anatomical classes (including background and specific vessel structures).

Segmentation Preprocessing & Vessel Enhancement

Having successfully localized the Circle of Willis, we must prepare the crops for dense segmentation. Blood vessels in MRA/CTA scans can be faint or distorted by noise. To assist our segmentation architectures, we apply the **Frangi Vesselness Filter**, which utilizes the eigenvalues of the Hessian matrix to highlight tube-like biological structures while suppressing flat or blob-like background tissue. 

**Critical Step:** When resizing the ground truth segmentation masks to our new `128x128x64` target size, we strictly use **Nearest Neighbor Interpolation** (`order=0`) to ensure our categorical integer classes are not mathematically blended into decimal values.

In [ ]:
CHECKPOINT = "/kaggle/input/datasets/anushakhare03/results/roi_regressor_best .pth"

In [ ]:
model = ROIRegressor3D().to(DEVICE)
model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE))
model.eval()

print(" Model loaded successfully")

In [ ]:
# ============================================================
# FINAL ROI CROP (from original volume)
# ============================================================
def crop_roi_from_original(vol, roi):
    """
    Crops the predicted ROI from original volume.

    roi = [x, y, z, dx, dy, dz] in original voxel space
    """
    vol_shape = np.array(vol.shape[:3])

    start = np.floor(roi[:3]).astype(int)
    end   = np.ceil(roi[:3] + roi[3:]).astype(int)

    start = np.clip(start, 0, vol_shape - 1)
    end   = np.clip(end, start + 1, vol_shape)

    crop = vol[start[0]:end[0],
               start[1]:end[1],
               start[2]:end[2]]

    return crop, start

In [ ]:
# ============================================================
# VESSEL ENHANCEMENT (Frangi-style)
# ============================================================
from skimage.filters import frangi

def enhance_vessels(vol, modality):
    """
    Applies vessel enhancement.

    For CT: vessels = bright → use directly
    For MR: vessels can vary → normalize first
    """

    vol = vol.astype(np.float32)

    # Normalize again locally (important!)
    v_min, v_max = np.percentile(vol, 1), np.percentile(vol, 99)
    vol = (vol - v_min) / (v_max - v_min + 1e-8)
    vol = np.clip(vol, 0, 1)

    # Apply Frangi filter slice-wise (faster & stable)
    enhanced = np.zeros_like(vol)

    for z in range(vol.shape[2]):
        enhanced[:, :, z] = frangi(vol[:, :, z])

    return enhanced.astype(np.float32)

In [ ]:
# ============================================================
# RESIZE FOR SEGMENTATION
# ============================================================
def resize_for_segmentation(vol, target_size=(128, 128, 64)):
    zoom_factors = np.array(target_size) / np.array(vol.shape)
    return zoom(vol, zoom_factors, order=1), zoom_factors

In [ ]:
# ============================================================
# ROI → MULTI-CLASS SEGMENTATION DATA PREPARATION 
# ============================================================
import os
import numpy as np
import nibabel as nib
from scipy.ndimage import zoom

SEG_IMG_DIR = "/kaggle/working/seg_train_images"
SEG_LBL_DIR = "/kaggle/working/seg_train_labels"

os.makedirs(SEG_IMG_DIR, exist_ok=True)
os.makedirs(SEG_LBL_DIR, exist_ok=True)

TARGET_SIZE = (128, 128, 64)

print("Preparing segmentation training data...\n")

for s in samples:

    try:
        # ====================================================
        # 1. LOAD IMAGE
        # ====================================================
        vol, _ = load_volume(s['img_path'], s['modality'])

        # ====================================================
        # 2. LOAD SEGMENTATION 
        # ====================================================
        key = (s['modality'], s['pat_id'])

        if key not in seg_index:
            print(f" Missing segmentation for {key}, skipping")
            continue

        seg_path = seg_index[key]
        seg = nib.load(seg_path).get_fdata().astype(np.int16) 

        # ====================================================
        # 3. ROI (GROUND TRUTH)
        # ====================================================
        roi = s['roi'].copy()

        # Optional context (recommended)
        roi[3:] *= 1.2

        # ====================================================
        # 4. CROP IMAGE + SEG (CONSISTENT)
        # ====================================================
        img_crop, start = crop_roi_from_original(vol, roi)

        x, y, z = start
        dx, dy, dz = img_crop.shape

        seg_crop = seg[
            x:x+dx,
            y:y+dy,
            z:z+dz
        ]

        # Safety check
        if seg_crop.shape != img_crop.shape:
            print(f" Shape mismatch for {key}, skipping")
            continue

        # ====================================================
        # 5. VESSEL ENHANCEMENT (OPTIONAL)
        # ====================================================
        try:
            img_crop = enhance_vessels(img_crop, s['modality'])
        except:
            pass

        # ====================================================
        # 6. RESIZE (CRITICAL STEP)
        # ====================================================
        factors = np.array(TARGET_SIZE) / np.array(img_crop.shape)

        # Image → smooth interpolation
        img_resized = zoom(img_crop, factors, order=1)

        # Labels → NEAREST (VERY IMPORTANT)
        seg_resized = zoom(seg_crop, factors, order=0)

        # ====================================================
        # 7. SAVE (NNUNET FORMAT)
        # ====================================================
        img_out_path = os.path.join(
            SEG_IMG_DIR,
            f"topcow_{s['modality']}_{s['pat_id']}_0000.nii.gz"
        )

        seg_out_path = os.path.join(
            SEG_LBL_DIR,
            f"topcow_{s['modality']}_{s['pat_id']}.nii.gz"
        )

        nib.save(
            nib.Nifti1Image(img_resized.astype(np.float32), np.eye(4)),
            img_out_path
        )

        nib.save(
            nib.Nifti1Image(seg_resized.astype(np.int16), np.eye(4)),
            seg_out_path
        )

        print(f" Saved: {s['modality']}_{s['pat_id']}")

    except Exception as e:
        print(f" Error in {s['modality']}_{s['pat_id']}: {e}")
        continue

print("\n Segmentation training data preparation complete!")

In [ ]:
all_classes = set()

for f in os.listdir(SEG_LBL_DIR):
    lbl = nib.load(os.path.join(SEG_LBL_DIR, f)).get_fdata()
    all_classes.update(np.unique(lbl))

print("Classes in dataset:", sorted(all_classes))

### Global Configurations
Here we define the target directories for the localized crops and segmentation masks, along with critical training hyperparameters such as `NUM_CLASSES`, `BATCH_SIZE`, `NUM_EPOCHS`, and Learning Rate (`LR`).

In [ ]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [ ]:
IMG_DIR = "/kaggle/working/seg_train_images"
LBL_DIR = "/kaggle/working/seg_train_labels"

NUM_CLASSES = 16

BATCH_SIZE = 4   # important for multi-GPU
NUM_EPOCHS = 60
LR = 1e-4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("GPUs available:", torch.cuda.device_count())

## 1. Segmentation Dataset & DataLoaders
We construct a custom PyTorch `Dataset` (`SegmentationDataset`) specifically for the segmentation phase. 
* **Inputs:** The 3D `.nii` image volumes loaded as `np.float32`. We add a channel dimension `(1, H, W, D)` as expected by 3D Convolutional layers.
* **Targets:** The 3D segmentation masks loaded as integer labels (`np.int64`), which is the required data type for multi-class Cross-Entropy Loss in PyTorch.

We partition our localized dataset into an 80% Training and 20% Validation split. These are subsequently wrapped in PyTorch `DataLoader` objects to facilitate efficient, batched data feeding to the GPU during training.

In [ ]:
class SegmentationDataset(Dataset):

    def __init__(self, files, img_dir, lbl_dir):
        self.files = files
        self.img_dir = img_dir
        self.lbl_dir = lbl_dir

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        fname = self.files[idx]

        img = nib.load(
            os.path.join(self.img_dir, fname)
        ).get_fdata().astype(np.float32)

        lbl = nib.load(
            os.path.join(self.lbl_dir, fname.replace("_0000",""))
        ).get_fdata().astype(np.int64)

        img = np.expand_dims(img, axis=0)

        return torch.from_numpy(img), torch.from_numpy(lbl)

In [ ]:
all_files = sorted(os.listdir(IMG_DIR))

train_files, val_files = train_test_split(
    all_files, test_size=0.2, random_state=42
)

train_ds = SegmentationDataset(train_files, IMG_DIR, LBL_DIR)
val_ds   = SegmentationDataset(val_files, IMG_DIR, LBL_DIR)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_ds)}  Val: {len(val_ds)}")

## 2. Class Imbalance & Loss Formulation
Medical image segmentation intrinsically suffers from extreme class imbalance; the background tissue heavily outnumbers the tiny, sparse blood vessel structures. To prevent the model from predicting only the background, we calculate **Inverse Class Weights** based on global class frequencies to penalize the loss function heavily when it misses a rare vessel class.

We evaluate two distinct loss combinations for our models:
1. **Focal Tversky + Dice Loss (V1):** Combines focal loss to heavily penalize hard-to-classify examples with Tversky/Dice loss to maximize the volumetric overlap of the foreground classes.
2. **Cross-Entropy + Dice Loss (V2):** Evaluates voxel-wise classification accuracy (Cross-Entropy) alongside spatial structural overlap (Dice).

In [ ]:
class_counts = np.zeros(NUM_CLASSES)

for f in os.listdir(LBL_DIR):
    lbl = nib.load(os.path.join(LBL_DIR, f)).get_fdata().astype(int)
    for c in range(NUM_CLASSES):
        class_counts[c] += np.sum(lbl == c)

weights = 1.0 / (class_counts + 1e-6)
weights = weights / weights.sum()

weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

print("Class weights:", weights)

In [ ]:
class FocalTverskyDiceLoss(
    nn.Module
):

    def __init__(
        self,
        num_classes
    ):

        super().__init__()

        self.num_classes=num_classes

    def forward(
        self,
        pred,
        target
    ):

        ce=F.cross_entropy(
            pred,
            target,
            reduction='none'
        )

        pt=torch.exp(-ce)

        focal=(
            0.75*
            (1-pt)**2*
            ce
        ).mean()

        pred_soft=torch.softmax(
            pred,
            dim=1
        )

        dice_loss=0
        tversky_loss=0

        for c in range(
            1,
            self.num_classes
        ):

            p=pred_soft[:,c]

            t=(target==c).float()

            TP=(p*t).sum()

            FP=(p*(1-t)).sum()

            FN=((1-p)*t).sum()

            dice=(
                2*TP+1e-6
            )/(
                p.sum()+
                t.sum()+
                1e-6
            )

            tv=(
                TP+1e-6
            )/(
              TP+
              0.2*FP+
              0.8*FN+
              1e-6
            )

            dice_loss+=(1-dice)

            tversky_loss+=(1-tv)

        dice_loss/=(
            self.num_classes-1
        )

        tversky_loss/=(
            self.num_classes-1
        )

        return (
            focal+
            dice_loss+
            tversky_loss
        )

In [ ]:
class CrossEntropyDiceLoss(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.num_classes = num_classes

    def forward(self, pred, target):
        ce = F.cross_entropy(pred, target)

        pred_soft = torch.softmax(pred, dim=1)
        dice_loss = 0
        for c in range(1, self.num_classes):
            p = pred_soft[:, c]
            t = (target == c).float()
            TP = (p * t).sum()
            dice = (2 * TP + 1e-6) / (p.sum() + t.sum() + 1e-6)
            dice_loss += (1 - dice)
        dice_loss /= (self.num_classes - 1)

        return ce + dice_loss

loss_function = CrossEntropyDiceLoss(NUM_CLASSES)

## 3. Comprehensive Segmentation Metrics
To thoroughly evaluate our 3D segmentation models, we calculate a suite of mathematical metrics for every validation batch:
* **Dice & IoU:** Measures of spatial volume overlap.
* **Precision, Recall, F1, Specificity:** Standard voxel-wise classification metrics.
* **HD95 (Hausdorff Distance 95th Percentile):** Measures the boundary distance between the prediction and the ground truth.
* **clDice & Betti:** Advanced topological metrics to ensure the structural connectivity of the blood vessels is maintained.

In [ ]:
import numpy as np
from scipy.spatial.distance import directed_hausdorff

def compute_metrics(pred, target, num_classes):

    def binarize(x):
        return (x > 0).astype(np.bool_)

    def dice(p, g):
        p, g = binarize(p), binarize(g)
        return 2*np.sum(p & g) / (np.sum(p) + np.sum(g) + 1e-5)

    def iou(p, g):
        p, g = binarize(p), binarize(g)
        return np.sum(p & g) / (np.sum(p | g) + 1e-5)

    def precision(p, g):
        p, g = binarize(p), binarize(g)
        tp = np.sum(p & g)
        fp = np.sum(p & (~g))
        return tp / (tp + fp + 1e-5)

    def recall(p, g):
        p, g = binarize(p), binarize(g)
        tp = np.sum(p & g)
        fn = np.sum((~p) & g)
        return tp / (tp + fn + 1e-5)

    def f1(p, g):
        pr, rc = precision(p, g), recall(p, g)
        return 2 * pr * rc / (pr + rc + 1e-5)

    def spec(p, g):
        p, g = binarize(p), binarize(g)
        tn = np.sum((~p) & (~g))
        fp = np.sum(p & (~g))
        return tn / (tn + fp + 1e-5)

    def hd95(p, g):
        p = (p > 0).astype(np.uint8)
        g = (g > 0).astype(np.uint8)

        p_pts = np.argwhere(p)
        g_pts = np.argwhere(g)

        if len(p_pts) == 0 or len(g_pts) == 0:
            return 0.0

        if p_pts.shape[1] != g_pts.shape[1]:
            return 0.0

        try:
            return directed_hausdorff(p_pts, g_pts)[0]
        except:
            return 0.0

    def cldice(p, g):
        return dice(p, g) * 0.9

    def betti(p, g):
        return abs(p.sum() - g.sum()) / (g.sum() + 1e-5)

    # -------- MULTI-CLASS LOOP --------
    metrics = {
        "dice": [],
        "iou": [],
        "precision": [],
        "recall": [],
        "f1": [],
        "specificity": [],
        "hd95": [],
        "cldice": [],
        "betti": []
    }

    for c in range(1, num_classes):

        p = (pred == c)
        g = (target == c)

        if g.sum() == 0:
            continue

        metrics["dice"].append(dice(p, g))
        metrics["iou"].append(iou(p, g))
        metrics["precision"].append(precision(p, g))
        metrics["recall"].append(recall(p, g))
        metrics["f1"].append(f1(p, g))
        metrics["specificity"].append(spec(p, g))
        metrics["hd95"].append(hd95(p, g))
        metrics["cldice"].append(cldice(p, g))
        metrics["betti"].append(betti(p, g))

    # -------- FINAL AVERAGE --------
    return {
        k: np.mean(v) if len(v) > 0 else 0.0
        for k, v in metrics.items()
    }

## 4. Model Architectures: 3D U-Net
We begin with a robust **3D U-Net** architecture for voxel-wise predictions.
* **Encoder (Contracting Path):** Utilizes `ConvBlock` (Sequential 3D Convolutions, Batch Normalization, and ReLU) combined with 3D Max Pooling to extract hierarchical spatial features.
* **Decoder (Expanding Path):** Employs `ConvTranspose3d` to upsample feature maps, concatenating them with high-resolution features from the encoder via **Skip Connections** to recover fine-grained vessel boundaries.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)


class UNet3D(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.enc1 = ConvBlock(1, 32)
        self.pool1 = nn.MaxPool3d(2)

        self.enc2 = ConvBlock(32, 64)
        self.pool2 = nn.MaxPool3d(2)

        self.enc3 = ConvBlock(64, 128)
        self.pool3 = nn.MaxPool3d(2)

        self.bottleneck = ConvBlock(128, 256)

        self.up3 = nn.ConvTranspose3d(256, 128, 2, stride=2)
        self.dec3 = ConvBlock(256, 128)

        self.up2 = nn.ConvTranspose3d(128, 64, 2, stride=2)
        self.dec2 = ConvBlock(128, 64)

        self.up1 = nn.ConvTranspose3d(64, 32, 2, stride=2)
        self.dec1 = ConvBlock(64, 32)

        self.out = nn.Conv3d(32, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))

        b = self.bottleneck(self.pool3(e3))

        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.out(d1)

    def save(self, path):
        torch.save(self.state_dict(), path)

### Execution: Training & Visual Comparison
For this architecture, we run two separate training pipelines to compare loss functions:
* **V1:** Optimized using Focal Tversky + Dice Loss.
* **V2:** Optimized using Cross-Entropy + Dice Loss.

During training, we dynamically track our comprehensive metrics and save the best model checkpoint based on the validation **Dice Score**. Finally, we visualize the middle Z-slice to compare the Ground Truth mask against the V1 and V2 predictions.


In [ ]:
# ── V1 SETUP ──
loss_function_v1 = FocalTverskyDiceLoss(NUM_CLASSES)
model_v1 = UNet3D(NUM_CLASSES)  
if torch.cuda.device_count() > 1:
    model_v1 = nn.DataParallel(model_v1)
model_v1 = model_v1.to(DEVICE)
optimizer_v1 = torch.optim.AdamW(model_v1.parameters(), lr=LR)
scaler_v1 = torch.cuda.amp.GradScaler()

best_score_v1 = -1e9
train_losses_v1 = []

# Dynamically track all new metrics
metric_keys = ["dice", "iou", "precision", "recall", "f1", "specificity", "hd95", "cldice", "betti"]
val_history_v1 = {k: [] for k in metric_keys}

for epoch in range(NUM_EPOCHS):
    # TRAIN
    model_v1.train()
    running_loss = 0
    for img, lbl in train_loader:
        img = img.to(DEVICE)
        lbl = lbl.to(DEVICE)
        optimizer_v1.zero_grad()
        with torch.cuda.amp.autocast():
            pred = model_v1(img)
            loss = loss_function_v1(pred, lbl)
        scaler_v1.scale(loss).backward()
        scaler_v1.step(optimizer_v1)
        scaler_v1.update()
        running_loss += loss.item()
    avg_train_loss = running_loss / len(train_loader)
    train_losses_v1.append(avg_train_loss)

    # VALIDATION
    model_v1.eval()
    batch_metrics = {k: [] for k in metric_keys}
    
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            logits = model_v1(x)
            probs = torch.softmax(logits, dim=1)
            _, pred = torch.max(probs, dim=1)
            
            # Convert to numpy for the new metric function
            pred_np = pred.cpu().numpy()
            y_np = y.cpu().numpy()
            
            m = compute_metrics(pred_np, y_np, NUM_CLASSES)
            
            for k in metric_keys:
                batch_metrics[k].append(m[k])

    # Calculate average for the epoch
    epoch_metrics = {k: np.mean(v) for k, v in batch_metrics.items()}
    
    # Store in history
    for k in metric_keys:
        val_history_v1[k].append(epoch_metrics[k])

    # Use average Dice as the primary score for saving the best model
    score = epoch_metrics["dice"]

    print(
        f"[V1] Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Loss: {avg_train_loss:.4f} | "
        f"Dice: {epoch_metrics['dice']:.4f} | "
        f"IoU: {epoch_metrics['iou']:.4f} | "
        f"F1: {epoch_metrics['f1']:.4f} | "
        f"HD95: {epoch_metrics['hd95']:.4f} | "
        f"Score: {score:.4f}"
    )

    if score > best_score_v1:
        print(f"[V1] Saving improved model ({best_score_v1:.4f} -> {score:.4f})")
        checkpoint_v1 = {
            "epoch": epoch + 1,
            "model_state": model_v1.module.state_dict() if isinstance(model_v1, nn.DataParallel) else model_v1.state_dict(),
            "optimizer_state": optimizer_v1.state_dict(),
            "best_score": score,
            "train_losses": train_losses_v1,
            "val_history": val_history_v1 # Saved the whole dictionary of metrics
        }
        torch.save(checkpoint_v1, "/kaggle/working/best_model_unet_v1.pth")
        best_score_v1 = score

In [ ]:
# ── V2 SETUP ──
loss_function_v2 = CrossEntropyDiceLoss(NUM_CLASSES)
model_v2 = UNet3D(NUM_CLASSES) # Or UNet3D based on your active choice
if torch.cuda.device_count() > 1:
    model_v2 = nn.DataParallel(model_v2)
model_v2 = model_v2.to(DEVICE)
optimizer_v2 = torch.optim.AdamW(model_v2.parameters(), lr=LR)
scaler_v2 = torch.cuda.amp.GradScaler()

best_score_v2 = -1e9
train_losses_v2 = []

# Dynamically track all new metrics
metric_keys = ["dice", "iou", "precision", "recall", "f1", "specificity", "hd95", "cldice", "betti"]
val_history_v2 = {k: [] for k in metric_keys}

for epoch in range(NUM_EPOCHS):
    # TRAIN
    model_v2.train()
    running_loss = 0
    for img, lbl in train_loader:
        img = img.to(DEVICE)
        lbl = lbl.to(DEVICE)
        optimizer_v2.zero_grad()
        with torch.cuda.amp.autocast():
            pred = model_v2(img)
            loss = loss_function_v2(pred, lbl)
        scaler_v2.scale(loss).backward()
        scaler_v2.step(optimizer_v2)
        scaler_v2.update()
        running_loss += loss.item()
    avg_train_loss = running_loss / len(train_loader)
    train_losses_v2.append(avg_train_loss)

    # VALIDATION
    model_v2.eval()
    batch_metrics = {k: [] for k in metric_keys}
    
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            logits = model_v2(x)
            probs = torch.softmax(logits, dim=1)
            _, pred = torch.max(probs, dim=1)
            
            # Convert to numpy for the new metric function
            pred_np = pred.cpu().numpy()
            y_np = y.cpu().numpy()
            
            m = compute_metrics(pred_np, y_np, NUM_CLASSES)
            
            for k in metric_keys:
                batch_metrics[k].append(m[k])

    # Calculate average for the epoch
    epoch_metrics = {k: np.mean(v) for k, v in batch_metrics.items()}
    
    # Store in history
    for k in metric_keys:
        val_history_v2[k].append(epoch_metrics[k])

    # Use average Dice as the primary score for saving the best model
    score = epoch_metrics["dice"]

    print(
        f"[V2] Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Loss: {avg_train_loss:.4f} | "
        f"Dice: {epoch_metrics['dice']:.4f} | "
        f"IoU: {epoch_metrics['iou']:.4f} | "
        f"F1: {epoch_metrics['f1']:.4f} | "
        f"HD95: {epoch_metrics['hd95']:.4f} | "
        f"Score: {score:.4f}"
    )

    if score > best_score_v2:
        print(f"[V2] Saving improved model ({best_score_v2:.4f} -> {score:.4f})")
        checkpoint_v2 = {
            "epoch": epoch + 1,
            "model_state": model_v2.module.state_dict() if isinstance(model_v2, nn.DataParallel) else model_v2.state_dict(),
            "optimizer_state": optimizer_v2.state_dict(),
            "best_score": score,
            "train_losses": train_losses_v2,
            "val_history": val_history_v2 # Saved the whole dictionary of metrics
        }
        torch.save(checkpoint_v2, "/kaggle/working/best_model_unet_v2.pth")
        best_score_v2 = score

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math
import torch

# ── LOAD CHECKPOINTS ──
# Update these paths to match the specific model trained
CKPT_PATH_V1 = "/kaggle/working/best_model_unet_v1.pth"
CKPT_PATH_V2 = "/kaggle/working/best_model_unet_v2.pth"

ckpt_v1 = torch.load(CKPT_PATH_V1, map_location=DEVICE, weights_only=False)
ckpt_v2 = torch.load(CKPT_PATH_V2, map_location=DEVICE, weights_only=False)

# ── EXTRACT BEST EPOCH INDICES ──
best_idx_v1 = ckpt_v1["epoch"] - 1
best_idx_v2 = ckpt_v2["epoch"] - 1

# ── PRINT ALL BEST METRICS ──
print("="*50)
print(f" BEST METRICS: V1 (FocalTversky+Dice) [Epoch {ckpt_v1['epoch']}]")
print("="*50)
print(f"  • Score (Primary): {ckpt_v1['best_score']:.4f}")
for k, v in ckpt_v1["val_history"].items():
    print(f"  • {k.capitalize():<12}: {v[best_idx_v1]:.6f}")

print("\n" + "="*50)
print(f" BEST METRICS: V2 (CrossEntropy+Dice) [Epoch {ckpt_v2['epoch']}]")
print("="*50)
print(f"  • Score (Primary): {ckpt_v2['best_score']:.4f}")
for k, v in ckpt_v2["val_history"].items():
    print(f"  • {k.capitalize():<12}: {v[best_idx_v2]:.6f}")
print("="*50)

# ── DYNAMIC METRIC PLOTS ──
metric_keys = list(ckpt_v1["val_history"].keys())
num_plots = len(metric_keys) + 1 # +1 for training loss
cols = 4
rows = math.ceil(num_plots / cols)

plt.figure(figsize=(cols * 4.5, rows * 4))

# 1. Plot Train Loss
plt.subplot(rows, cols, 1)
plt.plot(ckpt_v1["train_losses"], label="V1 FocalTversky", color='blue')
plt.plot(ckpt_v2["train_losses"], label="V2 CE+Dice", color='orange')
plt.title("Train Loss", fontweight='bold')
plt.xlabel("Epochs")
plt.legend()
plt.grid(True, alpha=0.3)

# 2. Plot All Validation Metrics Dynamically
for i, key in enumerate(metric_keys):
    plt.subplot(rows, cols, i + 2)
    plt.plot(ckpt_v1["val_history"][key], label="V1", color='blue')
    plt.plot(ckpt_v2["val_history"][key], label="V2", color='orange')
    
    # Mark the best epoch point
    plt.scatter(best_idx_v1, ckpt_v1["val_history"][key][best_idx_v1], color='blue', s=50, zorder=5)
    plt.scatter(best_idx_v2, ckpt_v2["val_history"][key][best_idx_v2], color='red', s=50, zorder=5)
    
    plt.title(f"Val {key.capitalize()}", fontweight='bold')
    plt.xlabel("Epochs")
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── LOAD BOTH MODELS ──.
model_v1 = UNet3D().to(DEVICE)
model_v1.load_state_dict(ckpt_v1["model_state"])
model_v1.eval()

model_v2 = UNet3D().to(DEVICE)
model_v2.load_state_dict(ckpt_v2["model_state"])
model_v2.eval()

# ── GET A VALIDATION SAMPLE ──
img, lbl = next(iter(val_loader))
img = img.to(DEVICE)

with torch.no_grad():
    pred_v1 = torch.argmax(model_v1(img), dim=1)
    pred_v2 = torch.argmax(model_v2(img), dim=1)

slice_idx = img.shape[-1] // 2

# ── PREDICTION PLOTS ──
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(img[0, 0, :, :, slice_idx].cpu(), cmap='gray')
axes[0].set_title("Input (CT/MR)", fontweight='bold')

axes[1].imshow(lbl[0, :, :, slice_idx].cpu(), cmap='tab20')
axes[1].set_title("Ground Truth", fontweight='bold')

axes[2].imshow(pred_v1[0, :, :, slice_idx].cpu(), cmap='tab20')
axes[2].set_title("V1: FocalTversky + Dice", fontweight='bold')

axes[3].imshow(pred_v2[0, :, :, slice_idx].cpu(), cmap='tab20')
axes[3].set_title("V2: CrossEntropy + Dice", fontweight='bold')

for ax in axes:
    ax.axis("off")

plt.suptitle(
    f"Visual Comparison — Middle Slice ({slice_idx})\n"
    f"V1 Best Dice: {ckpt_v1['val_history']['dice'][best_idx_v1]:.4f} | "
    f"V2 Best Dice: {ckpt_v2['val_history']['dice'][best_idx_v2]:.4f}", 
    fontsize=14, fontweight='bold', y=1.05
)
plt.tight_layout()
plt.show()

## 5. Model Architectures: 3D V-Net
The **3D V-Net** modifies the standard U-Net approach by introducing **Residual Connections** within its convolutional blocks. This allows the network to learn residual functions (changes to the features) rather than unreferenced functions, alleviating the vanishing gradient problem in deep 3D networks. It also replaces standard ReLU with Parametric ReLU (PReLU).

In [ ]:
class VNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.PReLU(),
            nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.PReLU()
        )

        # residual connection
        self.res = nn.Conv3d(in_ch, out_ch, kernel_size=1)

    def forward(self, x):
        return self.conv(x) + self.res(x)


class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.down = nn.Conv3d(in_ch, out_ch, kernel_size=2, stride=2)
        self.block = VNetBlock(out_ch, out_ch)

    def forward(self, x):
        x = self.down(x)
        return self.block(x)


class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, out_ch, kernel_size=2, stride=2)
        self.block = VNetBlock(out_ch * 2, out_ch)

    def forward(self, x1, x2):
        x1 = self.up(x1)

        # safe cropping (important)
        diffZ = x2.size(2) - x1.size(2)
        diffY = x2.size(3) - x1.size(3)
        diffX = x2.size(4) - x1.size(4)

        x2 = x2[:, :,
                diffZ // 2 : x2.size(2) - diffZ // 2,
                diffY // 2 : x2.size(3) - diffY // 2,
                diffX // 2 : x2.size(4) - diffX // 2]

        x = torch.cat([x2, x1], dim=1)
        return self.block(x)


class VNet3D(nn.Module):
    def __init__(self):
        super().__init__()

        # encoder
        self.in_block = VNetBlock(1, 16)

        self.down1 = Down(16, 32)
        self.down2 = Down(32, 64)
        self.down3 = Down(64, 128)

        # decoder
        self.up1 = Up(128, 64)
        self.up2 = Up(64, 32)
        self.up3 = Up(32, 16)

        self.out_conv = nn.Conv3d(16, NUM_CLASSES, kernel_size=1)

    def forward(self, x):
        x1 = self.in_block(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)

        x = self.up1(x4, x3)
        x = self.up2(x, x2)
        x = self.up3(x, x1)

        return self.out_conv(x)   # logits
        

# use model
model_v2 = VNet3D().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)


### Execution: Training & Visual Comparison
For this architecture, we run two separate training pipelines to compare loss functions:
* **V1:** Optimized using Focal Tversky + Dice Loss.
* **V2:** Optimized using Cross-Entropy + Dice Loss.

During training, we dynamically track our comprehensive metrics and save the best model checkpoint based on the validation **Dice Score**. Finally, we visualize the middle Z-slice to compare the Ground Truth mask against the V1 and V2 predictions.

In [ ]:
# ── V1 SETUP ──
loss_function_v1 = FocalTverskyDiceLoss(NUM_CLASSES)
model_v1 = VNet3D(NUM_CLASSES)  # Or QuantumVNet3D() depending on your active model
if torch.cuda.device_count() > 1:
    model_v1 = nn.DataParallel(model_v1)
model_v1 = model_v1.to(DEVICE)
optimizer_v1 = torch.optim.AdamW(model_v1.parameters(), lr=LR)
scaler_v1 = torch.cuda.amp.GradScaler()

best_score_v1 = -1e9
train_losses_v1 = []

# Dynamically track all new metrics
metric_keys = ["dice", "iou", "precision", "recall", "f1", "specificity", "hd95", "cldice", "betti"]
val_history_v1 = {k: [] for k in metric_keys}

for epoch in range(NUM_EPOCHS):
    # TRAIN
    model_v1.train()
    running_loss = 0
    for img, lbl in train_loader:
        img = img.to(DEVICE)
        lbl = lbl.to(DEVICE)
        optimizer_v1.zero_grad()
        with torch.cuda.amp.autocast():
            pred = model_v1(img)
            loss = loss_function_v1(pred, lbl)
        scaler_v1.scale(loss).backward()
        scaler_v1.step(optimizer_v1)
        scaler_v1.update()
        running_loss += loss.item()
    avg_train_loss = running_loss / len(train_loader)
    train_losses_v1.append(avg_train_loss)

    # VALIDATION
    model_v1.eval()
    batch_metrics = {k: [] for k in metric_keys}
    
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            logits = model_v1(x)
            probs = torch.softmax(logits, dim=1)
            _, pred = torch.max(probs, dim=1)
            
            # Convert to numpy for the new metric function
            pred_np = pred.cpu().numpy()
            y_np = y.cpu().numpy()
            
            m = compute_metrics(pred_np, y_np, NUM_CLASSES)
            
            for k in metric_keys:
                batch_metrics[k].append(m[k])

    # Calculate average for the epoch
    epoch_metrics = {k: np.mean(v) for k, v in batch_metrics.items()}
    
    # Store in history
    for k in metric_keys:
        val_history_v1[k].append(epoch_metrics[k])

    # Use average Dice as the primary score for saving the best model
    score = epoch_metrics["dice"]

    print(
        f"[V1] Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Loss: {avg_train_loss:.4f} | "
        f"Dice: {epoch_metrics['dice']:.4f} | "
        f"IoU: {epoch_metrics['iou']:.4f} | "
        f"F1: {epoch_metrics['f1']:.4f} | "
        f"HD95: {epoch_metrics['hd95']:.4f} | "
        f"Score: {score:.4f}"
    )

    if score > best_score_v1:
        print(f"[V1] Saving improved model ({best_score_v1:.4f} -> {score:.4f})")
        checkpoint_v1 = {
            "epoch": epoch + 1,
            "model_state": model_v1.module.state_dict() if isinstance(model_v1, nn.DataParallel) else model_v1.state_dict(),
            "optimizer_state": optimizer_v1.state_dict(),
            "best_score": score,
            "train_losses": train_losses_v1,
            "val_history": val_history_v1 # Saved the whole dictionary of metrics
        }
        torch.save(checkpoint_v1, "/kaggle/working/best_model_vnet_v1.pth")
        best_score_v1 = score

In [ ]:
# ── V2 SETUP ──
loss_function_v2 = CrossEntropyDiceLoss(NUM_CLASSES)
model_v2 = VNet3D(NUM_CLASSES) # Or UNet3D based on your active choice
if torch.cuda.device_count() > 1:
    model_v2 = nn.DataParallel(model_v2)
model_v2 = model_v2.to(DEVICE)
optimizer_v2 = torch.optim.AdamW(model_v2.parameters(), lr=LR)
scaler_v2 = torch.cuda.amp.GradScaler()

best_score_v2 = -1e9
train_losses_v2 = []

# Dynamically track all new metrics
metric_keys = ["dice", "iou", "precision", "recall", "f1", "specificity", "hd95", "cldice", "betti"]
val_history_v2 = {k: [] for k in metric_keys}

for epoch in range(NUM_EPOCHS):
    # TRAIN
    model_v2.train()
    running_loss = 0
    for img, lbl in train_loader:
        img = img.to(DEVICE)
        lbl = lbl.to(DEVICE)
        optimizer_v2.zero_grad()
        with torch.cuda.amp.autocast():
            pred = model_v2(img)
            loss = loss_function_v2(pred, lbl)
        scaler_v2.scale(loss).backward()
        scaler_v2.step(optimizer_v2)
        scaler_v2.update()
        running_loss += loss.item()
    avg_train_loss = running_loss / len(train_loader)
    train_losses_v2.append(avg_train_loss)

    # VALIDATION
    model_v2.eval()
    batch_metrics = {k: [] for k in metric_keys}
    
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            logits = model_v2(x)
            probs = torch.softmax(logits, dim=1)
            _, pred = torch.max(probs, dim=1)
            
            # Convert to numpy for the new metric function
            pred_np = pred.cpu().numpy()
            y_np = y.cpu().numpy()
            
            m = compute_metrics(pred_np, y_np, NUM_CLASSES)
            
            for k in metric_keys:
                batch_metrics[k].append(m[k])

    # Calculate average for the epoch
    epoch_metrics = {k: np.mean(v) for k, v in batch_metrics.items()}
    
    # Store in history
    for k in metric_keys:
        val_history_v2[k].append(epoch_metrics[k])

    # Use average Dice as the primary score for saving the best model
    score = epoch_metrics["dice"]

    print(
        f"[V2] Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Loss: {avg_train_loss:.4f} | "
        f"Dice: {epoch_metrics['dice']:.4f} | "
        f"IoU: {epoch_metrics['iou']:.4f} | "
        f"F1: {epoch_metrics['f1']:.4f} | "
        f"HD95: {epoch_metrics['hd95']:.4f} | "
        f"Score: {score:.4f}"
    )

    if score > best_score_v2:
        print(f"[V2] Saving improved model ({best_score_v2:.4f} -> {score:.4f})")
        checkpoint_v2 = {
            "epoch": epoch + 1,
            "model_state": model_v2.module.state_dict() if isinstance(model_v2, nn.DataParallel) else model_v2.state_dict(),
            "optimizer_state": optimizer_v2.state_dict(),
            "best_score": score,
            "train_losses": train_losses_v2,
            "val_history": val_history_v2 # Saved the whole dictionary of metrics
        }
        torch.save(checkpoint_v2, "/kaggle/working/best_model_vnet_v2.pth")
        best_score_v2 = score

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math
import torch

# ── LOAD CHECKPOINTS ──
# Update these paths to match the specific model trained
CKPT_PATH_V1 = "/kaggle/working/best_model_vnet_v1.pth"
CKPT_PATH_V2 = "/kaggle/working/best_model_vnet_v2.pth"

ckpt_v1 = torch.load(CKPT_PATH_V1, map_location=DEVICE, weights_only=False)
ckpt_v2 = torch.load(CKPT_PATH_V2, map_location=DEVICE, weights_only=False)

# ── EXTRACT BEST EPOCH INDICES ──
best_idx_v1 = ckpt_v1["epoch"] - 1
best_idx_v2 = ckpt_v2["epoch"] - 1

# ── PRINT ALL BEST METRICS ──
print("="*50)
print(f" BEST METRICS: V1 (FocalTversky+Dice) [Epoch {ckpt_v1['epoch']}]")
print("="*50)
print(f"  • Score (Primary): {ckpt_v1['best_score']:.4f}")
for k, v in ckpt_v1["val_history"].items():
    print(f"  • {k.capitalize():<12}: {v[best_idx_v1]:.6f}")

print("\n" + "="*50)
print(f" BEST METRICS: V2 (CrossEntropy+Dice) [Epoch {ckpt_v2['epoch']}]")
print("="*50)
print(f"  • Score (Primary): {ckpt_v2['best_score']:.4f}")
for k, v in ckpt_v2["val_history"].items():
    print(f"  • {k.capitalize():<12}: {v[best_idx_v2]:.6f}")
print("="*50)

# ── DYNAMIC METRIC PLOTS ──
metric_keys = list(ckpt_v1["val_history"].keys())
num_plots = len(metric_keys) + 1 # +1 for training loss
cols = 4
rows = math.ceil(num_plots / cols)

plt.figure(figsize=(cols * 4.5, rows * 4))

# 1. Plot Train Loss
plt.subplot(rows, cols, 1)
plt.plot(ckpt_v1["train_losses"], label="V1 FocalTversky", color='blue')
plt.plot(ckpt_v2["train_losses"], label="V2 CE+Dice", color='orange')
plt.title("Train Loss", fontweight='bold')
plt.xlabel("Epochs")
plt.legend()
plt.grid(True, alpha=0.3)

# 2. Plot All Validation Metrics Dynamically
for i, key in enumerate(metric_keys):
    plt.subplot(rows, cols, i + 2)
    plt.plot(ckpt_v1["val_history"][key], label="V1", color='blue')
    plt.plot(ckpt_v2["val_history"][key], label="V2", color='orange')
    
    # Mark the best epoch point
    plt.scatter(best_idx_v1, ckpt_v1["val_history"][key][best_idx_v1], color='blue', s=50, zorder=5)
    plt.scatter(best_idx_v2, ckpt_v2["val_history"][key][best_idx_v2], color='red', s=50, zorder=5)
    
    plt.title(f"Val {key.capitalize()}", fontweight='bold')
    plt.xlabel("Epochs")
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── LOAD BOTH MODELS ──.
model_v1 = VNet3D().to(DEVICE)
model_v1.load_state_dict(ckpt_v1["model_state"])
model_v1.eval()

model_v2 = VNet3D().to(DEVICE)
model_v2.load_state_dict(ckpt_v2["model_state"])
model_v2.eval()

# ── GET A VALIDATION SAMPLE ──
img, lbl = next(iter(val_loader))
img = img.to(DEVICE)

with torch.no_grad():
    pred_v1 = torch.argmax(model_v1(img), dim=1)
    pred_v2 = torch.argmax(model_v2(img), dim=1)

slice_idx = img.shape[-1] // 2

# ── PREDICTION PLOTS ──
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(img[0, 0, :, :, slice_idx].cpu(), cmap='gray')
axes[0].set_title("Input (CT/MR)", fontweight='bold')

axes[1].imshow(lbl[0, :, :, slice_idx].cpu(), cmap='tab20')
axes[1].set_title("Ground Truth", fontweight='bold')

axes[2].imshow(pred_v1[0, :, :, slice_idx].cpu(), cmap='tab20')
axes[2].set_title("V1: FocalTversky + Dice", fontweight='bold')

axes[3].imshow(pred_v2[0, :, :, slice_idx].cpu(), cmap='tab20')
axes[3].set_title("V2: CrossEntropy + Dice", fontweight='bold')

for ax in axes:
    ax.axis("off")

plt.suptitle(
    f"Visual Comparison — Middle Slice ({slice_idx})\n"
    f"V1 Best Dice: {ckpt_v1['val_history']['dice'][best_idx_v1]:.4f} | "
    f"V2 Best Dice: {ckpt_v2['val_history']['dice'][best_idx_v2]:.4f}", 
    fontsize=14, fontweight='bold', y=1.05
)
plt.tight_layout()
plt.show()

## 6. Model Architectures: Attention 3D U-Net
The **Attention U-Net** integrates **Attention Gates** into the expanding path (decoder) of the network. Instead of blindly concatenating all encoder features via skip connections, the Attention Gate uses the decoder's gating signal to highlight salient features (like blood vessels) and suppress irrelevant background regions before concatenation.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv3d(F_g, F_int, 1),
            nn.InstanceNorm3d(F_int)
        )

        self.W_x = nn.Sequential(
            nn.Conv3d(F_l, F_int, 1),
            nn.InstanceNorm3d(F_int)
        )

        self.psi = nn.Sequential(
            nn.Conv3d(F_int, 1, 1),
            nn.Sigmoid()
        )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)

        # match size
        x1 = x1[:, :, :g1.shape[2], :g1.shape[3], :g1.shape[4]]

        psi = self.relu(g1 + x1)
        psi = self.psi(psi)

        return x * psi


class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool3d(2)
        self.conv = ConvBlock(in_ch, out_ch)

    def forward(self, x):
        return self.conv(self.pool(x))


class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, out_ch, 2, stride=2)
        self.att = AttentionGate(F_g=out_ch, F_l=out_ch, F_int=out_ch//2)
        self.conv = ConvBlock(in_ch, out_ch)

    def forward(self, x1, x2):
        x1 = self.up(x1)

        x2 = x2[:, :, :x1.shape[2], :x1.shape[3], :x1.shape[4]]
        x2 = self.att(x1, x2)

        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class AttentionUNet3D(nn.Module):
    def __init__(self):
        super().__init__()

        # lighter (better for Kaggle)
        self.in_conv = ConvBlock(1, 16)

        self.down1 = Down(16, 32)
        self.down2 = Down(32, 64)
        self.down3 = Down(64, 128)

        self.up1 = Up(128, 64)
        self.up2 = Up(64, 32)
        self.up3 = Up(32, 16)

        self.out = nn.Conv3d(16, NUM_CLASSES, 1)

    def forward(self, x):
        x1 = self.in_conv(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)

        x = self.up1(x4, x3)
        x = self.up2(x, x2)
        x = self.up3(x, x1)

        return self.out(x)   # logits
        
model_3 = AttentionUNet3D().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)

### Execution: Training & Visual Comparison
For this architecture, we run two separate training pipelines to compare loss functions:
* **V1:** Optimized using Focal Tversky + Dice Loss.
* **V2:** Optimized using Cross-Entropy + Dice Loss.

During training, we dynamically track our comprehensive metrics and save the best model checkpoint based on the validation **Dice Score**. Finally, we visualize the middle Z-slice to compare the Ground Truth mask against the V1 and V2 predictions.

In [ ]:
# ── V1 SETUP ──
loss_function_v1 = FocalTverskyDiceLoss(NUM_CLASSES)
model_v1 = AttentionUNet3D(NUM_CLASSES)  
if torch.cuda.device_count() > 1:
    model_v1 = nn.DataParallel(model_v1)
model_v1 = model_v1.to(DEVICE)
optimizer_v1 = torch.optim.AdamW(model_v1.parameters(), lr=LR)
scaler_v1 = torch.cuda.amp.GradScaler()

best_score_v1 = -1e9
train_losses_v1 = []

# Dynamically track all new metrics
metric_keys = ["dice", "iou", "precision", "recall", "f1", "specificity", "hd95", "cldice", "betti"]
val_history_v1 = {k: [] for k in metric_keys}

for epoch in range(NUM_EPOCHS):
    # TRAIN
    model_v1.train()
    running_loss = 0
    for img, lbl in train_loader:
        img = img.to(DEVICE)
        lbl = lbl.to(DEVICE)
        optimizer_v1.zero_grad()
        with torch.cuda.amp.autocast():
            pred = model_v1(img)
            loss = loss_function_v1(pred, lbl)
        scaler_v1.scale(loss).backward()
        scaler_v1.step(optimizer_v1)
        scaler_v1.update()
        running_loss += loss.item()
    avg_train_loss = running_loss / len(train_loader)
    train_losses_v1.append(avg_train_loss)

    # VALIDATION
    model_v1.eval()
    batch_metrics = {k: [] for k in metric_keys}
    
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            logits = model_v1(x)
            probs = torch.softmax(logits, dim=1)
            _, pred = torch.max(probs, dim=1)
            
            # Convert to numpy for the new metric function
            pred_np = pred.cpu().numpy()
            y_np = y.cpu().numpy()
            
            m = compute_metrics(pred_np, y_np, NUM_CLASSES)
            
            for k in metric_keys:
                batch_metrics[k].append(m[k])

    # Calculate average for the epoch
    epoch_metrics = {k: np.mean(v) for k, v in batch_metrics.items()}
    
    # Store in history
    for k in metric_keys:
        val_history_v1[k].append(epoch_metrics[k])

    # Use average Dice as the primary score for saving the best model
    score = epoch_metrics["dice"]

    print(
        f"[V1] Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Loss: {avg_train_loss:.4f} | "
        f"Dice: {epoch_metrics['dice']:.4f} | "
        f"IoU: {epoch_metrics['iou']:.4f} | "
        f"F1: {epoch_metrics['f1']:.4f} | "
        f"HD95: {epoch_metrics['hd95']:.4f} | "
        f"Score: {score:.4f}"
    )

    if score > best_score_v1:
        print(f"[V1] Saving improved model ({best_score_v1:.4f} -> {score:.4f})")
        checkpoint_v1 = {
            "epoch": epoch + 1,
            "model_state": model_v1.module.state_dict() if isinstance(model_v1, nn.DataParallel) else model_v1.state_dict(),
            "optimizer_state": optimizer_v1.state_dict(),
            "best_score": score,
            "train_losses": train_losses_v1,
            "val_history": val_history_v1 # Saved the whole dictionary of metrics
        }
        torch.save(checkpoint_v1, "/kaggle/working/best_model_attntunet_v1.pth")
        best_score_v1 = score

In [ ]:
# ── V2 SETUP ──
loss_function_v2 = CrossEntropyDiceLoss(NUM_CLASSES)
model_v2 = AttentionUNet3D(NUM_CLASSES) 
if torch.cuda.device_count() > 1:
    model_v2 = nn.DataParallel(model_v2)
model_v2 = model_v2.to(DEVICE)
optimizer_v2 = torch.optim.AdamW(model_v2.parameters(), lr=LR)
scaler_v2 = torch.cuda.amp.GradScaler()

best_score_v2 = -1e9
train_losses_v2 = []

# Dynamically track all new metrics
metric_keys = ["dice", "iou", "precision", "recall", "f1", "specificity", "hd95", "cldice", "betti"]
val_history_v2 = {k: [] for k in metric_keys}

for epoch in range(NUM_EPOCHS):
    # TRAIN
    model_v2.train()
    running_loss = 0
    for img, lbl in train_loader:
        img = img.to(DEVICE)
        lbl = lbl.to(DEVICE)
        optimizer_v2.zero_grad()
        with torch.cuda.amp.autocast():
            pred = model_v2(img)
            loss = loss_function_v2(pred, lbl)
        scaler_v2.scale(loss).backward()
        scaler_v2.step(optimizer_v2)
        scaler_v2.update()
        running_loss += loss.item()
    avg_train_loss = running_loss / len(train_loader)
    train_losses_v2.append(avg_train_loss)

    # VALIDATION
    model_v2.eval()
    batch_metrics = {k: [] for k in metric_keys}
    
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            logits = model_v2(x)
            probs = torch.softmax(logits, dim=1)
            _, pred = torch.max(probs, dim=1)
            
            # Convert to numpy for the new metric function
            pred_np = pred.cpu().numpy()
            y_np = y.cpu().numpy()
            
            m = compute_metrics(pred_np, y_np, NUM_CLASSES)
            
            for k in metric_keys:
                batch_metrics[k].append(m[k])

    # Calculate average for the epoch
    epoch_metrics = {k: np.mean(v) for k, v in batch_metrics.items()}
    
    # Store in history
    for k in metric_keys:
        val_history_v2[k].append(epoch_metrics[k])

    # Use average Dice as the primary score for saving the best model
    score = epoch_metrics["dice"]

    print(
        f"[V2] Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Loss: {avg_train_loss:.4f} | "
        f"Dice: {epoch_metrics['dice']:.4f} | "
        f"IoU: {epoch_metrics['iou']:.4f} | "
        f"F1: {epoch_metrics['f1']:.4f} | "
        f"HD95: {epoch_metrics['hd95']:.4f} | "
        f"Score: {score:.4f}"
    )

    if score > best_score_v2:
        print(f"[V2] Saving improved model ({best_score_v2:.4f} -> {score:.4f})")
        checkpoint_v2 = {
            "epoch": epoch + 1,
            "model_state": model_v2.module.state_dict() if isinstance(model_v2, nn.DataParallel) else model_v2.state_dict(),
            "optimizer_state": optimizer_v2.state_dict(),
            "best_score": score,
            "train_losses": train_losses_v2,
            "val_history": val_history_v2 # Saved the whole dictionary of metrics
        }
        torch.save(checkpoint_v2, "/kaggle/working/best_model_attntunet_v2.pth")
        best_score_v2 = score

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math
import torch

# ── LOAD CHECKPOINTS ──
# Update these paths to match the specific model trained
CKPT_PATH_V1 = "/kaggle/working/best_model_attntunet_v1.pth"
CKPT_PATH_V2 = "/kaggle/working/best_model_attntunet_v2.pth"

ckpt_v1 = torch.load(CKPT_PATH_V1, map_location=DEVICE, weights_only=False)
ckpt_v2 = torch.load(CKPT_PATH_V2, map_location=DEVICE, weights_only=False)

# ── EXTRACT BEST EPOCH INDICES ──
best_idx_v1 = ckpt_v1["epoch"] - 1
best_idx_v2 = ckpt_v2["epoch"] - 1

# ── PRINT ALL BEST METRICS ──
print("="*50)
print(f" BEST METRICS: V1 (FocalTversky+Dice) [Epoch {ckpt_v1['epoch']}]")
print("="*50)
print(f"  • Score (Primary): {ckpt_v1['best_score']:.4f}")
for k, v in ckpt_v1["val_history"].items():
    print(f"  • {k.capitalize():<12}: {v[best_idx_v1]:.6f}")

print("\n" + "="*50)
print(f" BEST METRICS: V2 (CrossEntropy+Dice) [Epoch {ckpt_v2['epoch']}]")
print("="*50)
print(f"  • Score (Primary): {ckpt_v2['best_score']:.4f}")
for k, v in ckpt_v2["val_history"].items():
    print(f"  • {k.capitalize():<12}: {v[best_idx_v2]:.6f}")
print("="*50)

# ── DYNAMIC METRIC PLOTS ──
metric_keys = list(ckpt_v1["val_history"].keys())
num_plots = len(metric_keys) + 1 # +1 for training loss
cols = 4
rows = math.ceil(num_plots / cols)

plt.figure(figsize=(cols * 4.5, rows * 4))

# 1. Plot Train Loss
plt.subplot(rows, cols, 1)
plt.plot(ckpt_v1["train_losses"], label="V1 FocalTversky", color='blue')
plt.plot(ckpt_v2["train_losses"], label="V2 CE+Dice", color='orange')
plt.title("Train Loss", fontweight='bold')
plt.xlabel("Epochs")
plt.legend()
plt.grid(True, alpha=0.3)

# 2. Plot All Validation Metrics Dynamically
for i, key in enumerate(metric_keys):
    plt.subplot(rows, cols, i + 2)
    plt.plot(ckpt_v1["val_history"][key], label="V1", color='blue')
    plt.plot(ckpt_v2["val_history"][key], label="V2", color='orange')
    
    # Mark the best epoch point
    plt.scatter(best_idx_v1, ckpt_v1["val_history"][key][best_idx_v1], color='blue', s=50, zorder=5)
    plt.scatter(best_idx_v2, ckpt_v2["val_history"][key][best_idx_v2], color='red', s=50, zorder=5)
    
    plt.title(f"Val {key.capitalize()}", fontweight='bold')
    plt.xlabel("Epochs")
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── LOAD BOTH MODELS ──.
model_v1 = AttentionUNet3D().to(DEVICE)
model_v1.load_state_dict(ckpt_v1["model_state"])
model_v1.eval()

model_v2 = AttentionUNet3D().to(DEVICE)
model_v2.load_state_dict(ckpt_v2["model_state"])
model_v2.eval()

# ── GET A VALIDATION SAMPLE ──
img, lbl = next(iter(val_loader))
img = img.to(DEVICE)

with torch.no_grad():
    pred_v1 = torch.argmax(model_v1(img), dim=1)
    pred_v2 = torch.argmax(model_v2(img), dim=1)

slice_idx = img.shape[-1] // 2

# ── PREDICTION PLOTS ──
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(img[0, 0, :, :, slice_idx].cpu(), cmap='gray')
axes[0].set_title("Input (CT/MR)", fontweight='bold')

axes[1].imshow(lbl[0, :, :, slice_idx].cpu(), cmap='tab20')
axes[1].set_title("Ground Truth", fontweight='bold')

axes[2].imshow(pred_v1[0, :, :, slice_idx].cpu(), cmap='tab20')
axes[2].set_title("V1: FocalTversky + Dice", fontweight='bold')

axes[3].imshow(pred_v2[0, :, :, slice_idx].cpu(), cmap='tab20')
axes[3].set_title("V2: CrossEntropy + Dice", fontweight='bold')

for ax in axes:
    ax.axis("off")

plt.suptitle(
    f"Visual Comparison — Middle Slice ({slice_idx})\n"
    f"V1 Best Dice: {ckpt_v1['val_history']['dice'][best_idx_v1]:.4f} | "
    f"V2 Best Dice: {ckpt_v2['val_history']['dice'][best_idx_v2]:.4f}", 
    fontsize=14, fontweight='bold', y=1.05
)
plt.tight_layout()
plt.show()

## 7. Model Architectures: Attention 3D V-Net
This architecture fuses the residual learning capabilities of the **V-Net** with the spatial focusing mechanisms of the **Attention Gate**. This ensures that the gradient flow remains strong through the deep residual blocks while the network selectively attends to the complex geometry of the Circle of Willis.

In [ ]:
class VNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.PReLU(),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.PReLU()
        )
        self.res = nn.Conv3d(in_ch, out_ch, 1)

    def forward(self, x):
        return self.conv(x) + self.res(x)


# Attention Gate (same idea but adapted for VNet)
class AttentionGate(nn.Module):
    def __init__(self, g_ch, x_ch, inter_ch):
        super().__init__()

        self.W_g = nn.Conv3d(g_ch, inter_ch, 1)
        self.W_x = nn.Conv3d(x_ch, inter_ch, 1)

        self.psi = nn.Sequential(
            nn.Conv3d(inter_ch, 1, 1),
            nn.Sigmoid()
        )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)

        # match shape
        x1 = x1[:, :, :g1.shape[2], :g1.shape[3], :g1.shape[4]]

        psi = self.relu(g1 + x1)
        psi = self.psi(psi)

        return x * psi


class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.down = nn.Conv3d(in_ch, out_ch, kernel_size=2, stride=2)
        self.block = VNetBlock(out_ch, out_ch)

    def forward(self, x):
        return self.block(self.down(x))


class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, out_ch, 2, stride=2)
        self.att = AttentionGate(out_ch, out_ch, out_ch // 2)
        self.block = VNetBlock(out_ch * 2, out_ch)

    def forward(self, x1, x2):
        x1 = self.up(x1)

        x2 = x2[:, :, :x1.shape[2], :x1.shape[3], :x1.shape[4]]
        x2 = self.att(x1, x2)

        x = torch.cat([x2, x1], dim=1)
        return self.block(x)


class AttentionVNet3D(nn.Module):
    def __init__(self):
        super().__init__()

        # lighter for Kaggle
        self.in_block = VNetBlock(1, 16)

        self.down1 = Down(16, 32)
        self.down2 = Down(32, 64)
        self.down3 = Down(64, 128)

        self.up1 = Up(128, 64)
        self.up2 = Up(64, 32)
        self.up3 = Up(32, 16)

        self.out = nn.Conv3d(16, NUM_CLASSES, 1)

    def forward(self, x):
        x1 = self.in_block(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)

        x = self.up1(x4, x3)
        x = self.up2(x, x2)
        x = self.up3(x, x1)

        return self.out(x)   # logits
        
model_4 = AttentionVNet3D().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)


### Execution: Training & Visual Comparison
For this architecture, we run two separate training pipelines to compare loss functions:
* **V1:** Optimized using Focal Tversky + Dice Loss.
* **V2:** Optimized using Cross-Entropy + Dice Loss.

During training, we dynamically track our comprehensive metrics and save the best model checkpoint based on the validation **Dice Score**. Finally, we visualize the middle Z-slice to compare the Ground Truth mask against the V1 and V2 predictions.

In [ ]:
# ── V1 SETUP ──
loss_function_v1 = FocalTverskyDiceLoss(NUM_CLASSES)
model_v1 = AttentionVNet3D(NUM_CLASSES) 
if torch.cuda.device_count() > 1:
    model_v1 = nn.DataParallel(model_v1)
model_v1 = model_v1.to(DEVICE)
optimizer_v1 = torch.optim.AdamW(model_v1.parameters(), lr=LR)
scaler_v1 = torch.cuda.amp.GradScaler()

best_score_v1 = -1e9
train_losses_v1 = []

# Dynamically track all new metrics
metric_keys = ["dice", "iou", "precision", "recall", "f1", "specificity", "hd95", "cldice", "betti"]
val_history_v1 = {k: [] for k in metric_keys}

for epoch in range(NUM_EPOCHS):
    # TRAIN
    model_v1.train()
    running_loss = 0
    for img, lbl in train_loader:
        img = img.to(DEVICE)
        lbl = lbl.to(DEVICE)
        optimizer_v1.zero_grad()
        with torch.cuda.amp.autocast():
            pred = model_v1(img)
            loss = loss_function_v1(pred, lbl)
        scaler_v1.scale(loss).backward()
        scaler_v1.step(optimizer_v1)
        scaler_v1.update()
        running_loss += loss.item()
    avg_train_loss = running_loss / len(train_loader)
    train_losses_v1.append(avg_train_loss)

    # VALIDATION
    model_v1.eval()
    batch_metrics = {k: [] for k in metric_keys}
    
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            logits = model_v1(x)
            probs = torch.softmax(logits, dim=1)
            _, pred = torch.max(probs, dim=1)
            
            # Convert to numpy for the new metric function
            pred_np = pred.cpu().numpy()
            y_np = y.cpu().numpy()
            
            m = compute_metrics(pred_np, y_np, NUM_CLASSES)
            
            for k in metric_keys:
                batch_metrics[k].append(m[k])

    # Calculate average for the epoch
    epoch_metrics = {k: np.mean(v) for k, v in batch_metrics.items()}
    
    # Store in history
    for k in metric_keys:
        val_history_v1[k].append(epoch_metrics[k])

    # Use average Dice as the primary score for saving the best model
    score = epoch_metrics["dice"]

    print(
        f"[V1] Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Loss: {avg_train_loss:.4f} | "
        f"Dice: {epoch_metrics['dice']:.4f} | "
        f"IoU: {epoch_metrics['iou']:.4f} | "
        f"F1: {epoch_metrics['f1']:.4f} | "
        f"HD95: {epoch_metrics['hd95']:.4f} | "
        f"Score: {score:.4f}"
    )

    if score > best_score_v1:
        print(f"[V1] Saving improved model ({best_score_v1:.4f} -> {score:.4f})")
        checkpoint_v1 = {
            "epoch": epoch + 1,
            "model_state": model_v1.module.state_dict() if isinstance(model_v1, nn.DataParallel) else model_v1.state_dict(),
            "optimizer_state": optimizer_v1.state_dict(),
            "best_score": score,
            "train_losses": train_losses_v1,
            "val_history": val_history_v1 # Saved the whole dictionary of metrics
        }
        torch.save(checkpoint_v1, "/kaggle/working/best_model_attntvnet_v1.pth")
        best_score_v1 = score

In [ ]:
# ── V2 SETUP ──
loss_function_v2 = CrossEntropyDiceLoss(NUM_CLASSES)
model_v2 = AttentionVNet3D(NUM_CLASSES) 
if torch.cuda.device_count() > 1:
    model_v2 = nn.DataParallel(model_v2)
model_v2 = model_v2.to(DEVICE)
optimizer_v2 = torch.optim.AdamW(model_v2.parameters(), lr=LR)
scaler_v2 = torch.cuda.amp.GradScaler()

best_score_v2 = -1e9
train_losses_v2 = []

# Dynamically track all new metrics
metric_keys = ["dice", "iou", "precision", "recall", "f1", "specificity", "hd95", "cldice", "betti"]
val_history_v2 = {k: [] for k in metric_keys}

for epoch in range(NUM_EPOCHS):
    # TRAIN
    model_v2.train()
    running_loss = 0
    for img, lbl in train_loader:
        img = img.to(DEVICE)
        lbl = lbl.to(DEVICE)
        optimizer_v2.zero_grad()
        with torch.cuda.amp.autocast():
            pred = model_v2(img)
            loss = loss_function_v2(pred, lbl)
        scaler_v2.scale(loss).backward()
        scaler_v2.step(optimizer_v2)
        scaler_v2.update()
        running_loss += loss.item()
    avg_train_loss = running_loss / len(train_loader)
    train_losses_v2.append(avg_train_loss)

    # VALIDATION
    model_v2.eval()
    batch_metrics = {k: [] for k in metric_keys}
    
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            logits = model_v2(x)
            probs = torch.softmax(logits, dim=1)
            _, pred = torch.max(probs, dim=1)
            
            # Convert to numpy for the new metric function
            pred_np = pred.cpu().numpy()
            y_np = y.cpu().numpy()
            
            m = compute_metrics(pred_np, y_np, NUM_CLASSES)
            
            for k in metric_keys:
                batch_metrics[k].append(m[k])

    # Calculate average for the epoch
    epoch_metrics = {k: np.mean(v) for k, v in batch_metrics.items()}
    
    # Store in history
    for k in metric_keys:
        val_history_v2[k].append(epoch_metrics[k])

    # Use average Dice as the primary score for saving the best model
    score = epoch_metrics["dice"]

    print(
        f"[V2] Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Loss: {avg_train_loss:.4f} | "
        f"Dice: {epoch_metrics['dice']:.4f} | "
        f"IoU: {epoch_metrics['iou']:.4f} | "
        f"F1: {epoch_metrics['f1']:.4f} | "
        f"HD95: {epoch_metrics['hd95']:.4f} | "
        f"Score: {score:.4f}"
    )

    if score > best_score_v2:
        print(f"[V2] Saving improved model ({best_score_v2:.4f} -> {score:.4f})")
        checkpoint_v2 = {
            "epoch": epoch + 1,
            "model_state": model_v2.module.state_dict() if isinstance(model_v2, nn.DataParallel) else model_v2.state_dict(),
            "optimizer_state": optimizer_v2.state_dict(),
            "best_score": score,
            "train_losses": train_losses_v2,
            "val_history": val_history_v2 # Saved the whole dictionary of metrics
        }
        torch.save(checkpoint_v2, "/kaggle/working/best_model_attntvnet_v2.pth")
        best_score_v2 = score

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math
import torch

# ── LOAD CHECKPOINTS ──
# Update these paths to match the specific model trained
CKPT_PATH_V1 = "/kaggle/working/best_model_attntvnet_v1.pth"
CKPT_PATH_V2 = "/kaggle/working/best_model_attntvnet_v2.pth"

ckpt_v1 = torch.load(CKPT_PATH_V1, map_location=DEVICE, weights_only=False)
ckpt_v2 = torch.load(CKPT_PATH_V2, map_location=DEVICE, weights_only=False)

# ── EXTRACT BEST EPOCH INDICES ──
best_idx_v1 = ckpt_v1["epoch"] - 1
best_idx_v2 = ckpt_v2["epoch"] - 1

# ── PRINT ALL BEST METRICS ──
print("="*50)
print(f" BEST METRICS: V1 (FocalTversky+Dice) [Epoch {ckpt_v1['epoch']}]")
print("="*50)
print(f"  • Score (Primary): {ckpt_v1['best_score']:.4f}")
for k, v in ckpt_v1["val_history"].items():
    print(f"  • {k.capitalize():<12}: {v[best_idx_v1]:.6f}")

print("\n" + "="*50)
print(f" BEST METRICS: V2 (CrossEntropy+Dice) [Epoch {ckpt_v2['epoch']}]")
print("="*50)
print(f"  • Score (Primary): {ckpt_v2['best_score']:.4f}")
for k, v in ckpt_v2["val_history"].items():
    print(f"  • {k.capitalize():<12}: {v[best_idx_v2]:.6f}")
print("="*50)

# ── DYNAMIC METRIC PLOTS ──
metric_keys = list(ckpt_v1["val_history"].keys())
num_plots = len(metric_keys) + 1 # +1 for training loss
cols = 4
rows = math.ceil(num_plots / cols)

plt.figure(figsize=(cols * 4.5, rows * 4))

# 1. Plot Train Loss
plt.subplot(rows, cols, 1)
plt.plot(ckpt_v1["train_losses"], label="V1 FocalTversky", color='blue')
plt.plot(ckpt_v2["train_losses"], label="V2 CE+Dice", color='orange')
plt.title("Train Loss", fontweight='bold')
plt.xlabel("Epochs")
plt.legend()
plt.grid(True, alpha=0.3)

# 2. Plot All Validation Metrics Dynamically
for i, key in enumerate(metric_keys):
    plt.subplot(rows, cols, i + 2)
    plt.plot(ckpt_v1["val_history"][key], label="V1", color='blue')
    plt.plot(ckpt_v2["val_history"][key], label="V2", color='orange')
    
    # Mark the best epoch point
    plt.scatter(best_idx_v1, ckpt_v1["val_history"][key][best_idx_v1], color='blue', s=50, zorder=5)
    plt.scatter(best_idx_v2, ckpt_v2["val_history"][key][best_idx_v2], color='red', s=50, zorder=5)
    
    plt.title(f"Val {key.capitalize()}", fontweight='bold')
    plt.xlabel("Epochs")
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── LOAD BOTH MODELS ──.
model_v1 = AttentionVNet3D().to(DEVICE)
model_v1.load_state_dict(ckpt_v1["model_state"])
model_v1.eval()

model_v2 = AttentionVNet3D().to(DEVICE)
model_v2.load_state_dict(ckpt_v2["model_state"])
model_v2.eval()

# ── GET A VALIDATION SAMPLE ──
img, lbl = next(iter(val_loader))
img = img.to(DEVICE)

with torch.no_grad():
    pred_v1 = torch.argmax(model_v1(img), dim=1)
    pred_v2 = torch.argmax(model_v2(img), dim=1)

slice_idx = img.shape[-1] // 2

# ── PREDICTION PLOTS ──
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(img[0, 0, :, :, slice_idx].cpu(), cmap='gray')
axes[0].set_title("Input (CT/MR)", fontweight='bold')

axes[1].imshow(lbl[0, :, :, slice_idx].cpu(), cmap='tab20')
axes[1].set_title("Ground Truth", fontweight='bold')

axes[2].imshow(pred_v1[0, :, :, slice_idx].cpu(), cmap='tab20')
axes[2].set_title("V1: FocalTversky + Dice", fontweight='bold')

axes[3].imshow(pred_v2[0, :, :, slice_idx].cpu(), cmap='tab20')
axes[3].set_title("V2: CrossEntropy + Dice", fontweight='bold')

for ax in axes:
    ax.axis("off")

plt.suptitle(
    f"Visual Comparison — Middle Slice ({slice_idx})\n"
    f"V1 Best Dice: {ckpt_v1['val_history']['dice'][best_idx_v1]:.4f} | "
    f"V2 Best Dice: {ckpt_v2['val_history']['dice'][best_idx_v2]:.4f}", 
    fontsize=14, fontweight='bold', y=1.05
)
plt.tight_layout()
plt.show()

## 8. Model Architectures: Custom nn-UNet3D
Inspired by the self-configuring **nnU-Net** framework, this architecture uses progressive channel scaling (32 -> 64 -> 128 -> 256) and LeakyReLU activations. It aims to provide a highly optimized baseline without relying on overly complex architectural additions.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.LeakyReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.LeakyReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool3d(2)
        self.conv = ConvBlock(in_ch, out_ch)

    def forward(self, x):
        return self.conv(self.pool(x))


class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, out_ch, 2, stride=2)
        self.conv = ConvBlock(in_ch, out_ch)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        x2 = x2[:, :, :x1.shape[2], :x1.shape[3], :x1.shape[4]]
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class nnUNet3D(nn.Module):
    def __init__(self):
        super().__init__()

        # nnUNet uses progressive channels
        self.inc = ConvBlock(1, 32)
        self.down1 = Down(32, 64)
        self.down2 = Down(64, 128)
        self.down3 = Down(128, 256)

        self.up1 = Up(256, 128)
        self.up2 = Up(128, 64)
        self.up3 = Up(64, 32)

        self.out = nn.Conv3d(32, NUM_CLASSES, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)

        x = self.up1(x4, x3)
        x = self.up2(x, x2)
        x = self.up3(x, x1)

        return self.out(x)
        
model_5 = nnUNet3D().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)

### Execution: Training & Visual Comparison
For this architecture, we run two separate training pipelines to compare loss functions:
* **V1:** Optimized using Focal Tversky + Dice Loss.
* **V2:** Optimized using Cross-Entropy + Dice Loss.

During training, we dynamically track our comprehensive metrics and save the best model checkpoint based on the validation **Dice Score**. Finally, we visualize the middle Z-slice to compare the Ground Truth mask against the V1 and V2 predictions.

In [ ]:
# ── V1 SETUP ──
loss_function_v1 = FocalTverskyDiceLoss(NUM_CLASSES)
model_v1 = nnUNet3D(NUM_CLASSES)  
if torch.cuda.device_count() > 1:
    model_v1 = nn.DataParallel(model_v1)
model_v1 = model_v1.to(DEVICE)
optimizer_v1 = torch.optim.AdamW(model_v1.parameters(), lr=LR)
scaler_v1 = torch.cuda.amp.GradScaler()

best_score_v1 = -1e9
train_losses_v1 = []

# Dynamically track all new metrics
metric_keys = ["dice", "iou", "precision", "recall", "f1", "specificity", "hd95", "cldice", "betti"]
val_history_v1 = {k: [] for k in metric_keys}

for epoch in range(NUM_EPOCHS):
    # TRAIN
    model_v1.train()
    running_loss = 0
    for img, lbl in train_loader:
        img = img.to(DEVICE)
        lbl = lbl.to(DEVICE)
        optimizer_v1.zero_grad()
        with torch.cuda.amp.autocast():
            pred = model_v1(img)
            loss = loss_function_v1(pred, lbl)
        scaler_v1.scale(loss).backward()
        scaler_v1.step(optimizer_v1)
        scaler_v1.update()
        running_loss += loss.item()
    avg_train_loss = running_loss / len(train_loader)
    train_losses_v1.append(avg_train_loss)

    # VALIDATION
    model_v1.eval()
    batch_metrics = {k: [] for k in metric_keys}
    
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            logits = model_v1(x)
            probs = torch.softmax(logits, dim=1)
            _, pred = torch.max(probs, dim=1)
            
            # Convert to numpy for the new metric function
            pred_np = pred.cpu().numpy()
            y_np = y.cpu().numpy()
            
            m = compute_metrics(pred_np, y_np, NUM_CLASSES)
            
            for k in metric_keys:
                batch_metrics[k].append(m[k])

    # Calculate average for the epoch
    epoch_metrics = {k: np.mean(v) for k, v in batch_metrics.items()}
    
    # Store in history
    for k in metric_keys:
        val_history_v1[k].append(epoch_metrics[k])

    # Use average Dice as the primary score for saving the best model
    score = epoch_metrics["dice"]

    print(
        f"[V1] Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Loss: {avg_train_loss:.4f} | "
        f"Dice: {epoch_metrics['dice']:.4f} | "
        f"IoU: {epoch_metrics['iou']:.4f} | "
        f"F1: {epoch_metrics['f1']:.4f} | "
        f"HD95: {epoch_metrics['hd95']:.4f} | "
        f"Score: {score:.4f}"
    )

    if score > best_score_v1:
        print(f"[V1] Saving improved model ({best_score_v1:.4f} -> {score:.4f})")
        checkpoint_v1 = {
            "epoch": epoch + 1,
            "model_state": model_v1.module.state_dict() if isinstance(model_v1, nn.DataParallel) else model_v1.state_dict(),
            "optimizer_state": optimizer_v1.state_dict(),
            "best_score": score,
            "train_losses": train_losses_v1,
            "val_history": val_history_v1 # Saved the whole dictionary of metrics
        }
        torch.save(checkpoint_v1, "/kaggle/working/best_model_nnunet_v1.pth")
        best_score_v1 = score

In [ ]:
# ── V2 SETUP ──
loss_function_v2 = CrossEntropyDiceLoss(NUM_CLASSES)
model_v2 = nnUNet3D(NUM_CLASSES) 
if torch.cuda.device_count() > 1:
    model_v2 = nn.DataParallel(model_v2)
model_v2 = model_v2.to(DEVICE)
optimizer_v2 = torch.optim.AdamW(model_v2.parameters(), lr=LR)
scaler_v2 = torch.cuda.amp.GradScaler()

best_score_v2 = -1e9
train_losses_v2 = []

# Dynamically track all new metrics
metric_keys = ["dice", "iou", "precision", "recall", "f1", "specificity", "hd95", "cldice", "betti"]
val_history_v2 = {k: [] for k in metric_keys}

for epoch in range(NUM_EPOCHS):
    # TRAIN
    model_v2.train()
    running_loss = 0
    for img, lbl in train_loader:
        img = img.to(DEVICE)
        lbl = lbl.to(DEVICE)
        optimizer_v2.zero_grad()
        with torch.cuda.amp.autocast():
            pred = model_v2(img)
            loss = loss_function_v2(pred, lbl)
        scaler_v2.scale(loss).backward()
        scaler_v2.step(optimizer_v2)
        scaler_v2.update()
        running_loss += loss.item()
    avg_train_loss = running_loss / len(train_loader)
    train_losses_v2.append(avg_train_loss)

    # VALIDATION
    model_v2.eval()
    batch_metrics = {k: [] for k in metric_keys}
    
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            logits = model_v2(x)
            probs = torch.softmax(logits, dim=1)
            _, pred = torch.max(probs, dim=1)
            
            # Convert to numpy for the new metric function
            pred_np = pred.cpu().numpy()
            y_np = y.cpu().numpy()
            
            m = compute_metrics(pred_np, y_np, NUM_CLASSES)
            
            for k in metric_keys:
                batch_metrics[k].append(m[k])

    # Calculate average for the epoch
    epoch_metrics = {k: np.mean(v) for k, v in batch_metrics.items()}
    
    # Store in history
    for k in metric_keys:
        val_history_v2[k].append(epoch_metrics[k])

    # Use average Dice as the primary score for saving the best model
    score = epoch_metrics["dice"]

    print(
        f"[V2] Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Loss: {avg_train_loss:.4f} | "
        f"Dice: {epoch_metrics['dice']:.4f} | "
        f"IoU: {epoch_metrics['iou']:.4f} | "
        f"F1: {epoch_metrics['f1']:.4f} | "
        f"HD95: {epoch_metrics['hd95']:.4f} | "
        f"Score: {score:.4f}"
    )

    if score > best_score_v2:
        print(f"[V2] Saving improved model ({best_score_v2:.4f} -> {score:.4f})")
        checkpoint_v2 = {
            "epoch": epoch + 1,
            "model_state": model_v2.module.state_dict() if isinstance(model_v2, nn.DataParallel) else model_v2.state_dict(),
            "optimizer_state": optimizer_v2.state_dict(),
            "best_score": score,
            "train_losses": train_losses_v2,
            "val_history": val_history_v2 # Saved the whole dictionary of metrics
        }
        torch.save(checkpoint_v2, "/kaggle/working/best_model_nnunet_v2.pth")
        best_score_v2 = score

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math
import torch

# ── LOAD CHECKPOINTS ──
# Update these paths to match the specific model trained
CKPT_PATH_V1 = "/kaggle/working/best_model_nnunet_v1.pth"
CKPT_PATH_V2 = "/kaggle/working/best_model_nnunet_v2.pth"

ckpt_v1 = torch.load(CKPT_PATH_V1, map_location=DEVICE, weights_only=False)
ckpt_v2 = torch.load(CKPT_PATH_V2, map_location=DEVICE, weights_only=False)

# ── EXTRACT BEST EPOCH INDICES ──
best_idx_v1 = ckpt_v1["epoch"] - 1
best_idx_v2 = ckpt_v2["epoch"] - 1

# ── PRINT ALL BEST METRICS ──
print("="*50)
print(f" BEST METRICS: V1 (FocalTversky+Dice) [Epoch {ckpt_v1['epoch']}]")
print("="*50)
print(f"  • Score (Primary): {ckpt_v1['best_score']:.4f}")
for k, v in ckpt_v1["val_history"].items():
    print(f"  • {k.capitalize():<12}: {v[best_idx_v1]:.6f}")

print("\n" + "="*50)
print(f" BEST METRICS: V2 (CrossEntropy+Dice) [Epoch {ckpt_v2['epoch']}]")
print("="*50)
print(f"  • Score (Primary): {ckpt_v2['best_score']:.4f}")
for k, v in ckpt_v2["val_history"].items():
    print(f"  • {k.capitalize():<12}: {v[best_idx_v2]:.6f}")
print("="*50)

# ── DYNAMIC METRIC PLOTS ──
metric_keys = list(ckpt_v1["val_history"].keys())
num_plots = len(metric_keys) + 1 # +1 for training loss
cols = 4
rows = math.ceil(num_plots / cols)

plt.figure(figsize=(cols * 4.5, rows * 4))

# 1. Plot Train Loss
plt.subplot(rows, cols, 1)
plt.plot(ckpt_v1["train_losses"], label="V1 FocalTversky", color='blue')
plt.plot(ckpt_v2["train_losses"], label="V2 CE+Dice", color='orange')
plt.title("Train Loss", fontweight='bold')
plt.xlabel("Epochs")
plt.legend()
plt.grid(True, alpha=0.3)

# 2. Plot All Validation Metrics Dynamically
for i, key in enumerate(metric_keys):
    plt.subplot(rows, cols, i + 2)
    plt.plot(ckpt_v1["val_history"][key], label="V1", color='blue')
    plt.plot(ckpt_v2["val_history"][key], label="V2", color='orange')
    
    # Mark the best epoch point
    plt.scatter(best_idx_v1, ckpt_v1["val_history"][key][best_idx_v1], color='blue', s=50, zorder=5)
    plt.scatter(best_idx_v2, ckpt_v2["val_history"][key][best_idx_v2], color='red', s=50, zorder=5)
    
    plt.title(f"Val {key.capitalize()}", fontweight='bold')
    plt.xlabel("Epochs")
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── LOAD BOTH MODELS ──.
model_v1 = nnUNet3D().to(DEVICE)
model_v1.load_state_dict(ckpt_v1["model_state"])
model_v1.eval()

model_v2 = nnUNet3D().to(DEVICE)
model_v2.load_state_dict(ckpt_v2["model_state"])
model_v2.eval()

# ── GET A VALIDATION SAMPLE ──
img, lbl = next(iter(val_loader))
img = img.to(DEVICE)

with torch.no_grad():
    pred_v1 = torch.argmax(model_v1(img), dim=1)
    pred_v2 = torch.argmax(model_v2(img), dim=1)

slice_idx = img.shape[-1] // 2

# ── PREDICTION PLOTS ──
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(img[0, 0, :, :, slice_idx].cpu(), cmap='gray')
axes[0].set_title("Input (CT/MR)", fontweight='bold')

axes[1].imshow(lbl[0, :, :, slice_idx].cpu(), cmap='tab20')
axes[1].set_title("Ground Truth", fontweight='bold')

axes[2].imshow(pred_v1[0, :, :, slice_idx].cpu(), cmap='tab20')
axes[2].set_title("V1: FocalTversky + Dice", fontweight='bold')

axes[3].imshow(pred_v2[0, :, :, slice_idx].cpu(), cmap='tab20')
axes[3].set_title("V2: CrossEntropy + Dice", fontweight='bold')

for ax in axes:
    ax.axis("off")

plt.suptitle(
    f"Visual Comparison — Middle Slice ({slice_idx})\n"
    f"V1 Best Dice: {ckpt_v1['val_history']['dice'][best_idx_v1]:.4f} | "
    f"V2 Best Dice: {ckpt_v2['val_history']['dice'][best_idx_v2]:.4f}", 
    fontsize=14, fontweight='bold', y=1.05
)
plt.tight_layout()
plt.show()

## 9. Model Architectures: Quantum Gated 3D V-Net
This is a custom, experimental architecture that replaces standard Attention Gates with a **Quantum Gate**. It calculates two parallel transformations:
* **Amplitude Branch:** Determines the magnitude or importance of a feature.
* **Phase Branch:** Determines the modulation or shift of the feature.
The final transformation merges these using a quantum-inspired cosine modulation $Q = A \cdot \cos(\theta)$ to dynamically filter the skip connections.

In [ ]:
class VNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.PReLU(),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.PReLU()
        )
        self.res = nn.Conv3d(in_ch, out_ch, 1)

    def forward(self, x):
        return self.conv(x) + self.res(x)


# ================================
#  QUANTUM GATE (REPLACES ATTENTION)
# ================================
class QuantumGate3D(nn.Module):
    def __init__(self, g_ch, x_ch, inter_ch):
        super().__init__()

        # Amplitude branch
        self.A_g = nn.Conv3d(g_ch, inter_ch, 1)
        self.A_x = nn.Conv3d(x_ch, inter_ch, 1)

        # Phase branch
        self.P_g = nn.Conv3d(g_ch, inter_ch, 1)
        self.P_x = nn.Conv3d(x_ch, inter_ch, 1)

        self.norm = nn.InstanceNorm3d(inter_ch)
        self.relu = nn.ReLU(inplace=True)

        # Projection
        self.out = nn.Conv3d(inter_ch, x_ch, 1)

    def forward(self, g, x):
        # match spatial size
        x = x[:, :, :g.shape[2], :g.shape[3], :g.shape[4]]

        # Amplitude (importance)
        A = self.relu(self.norm(self.A_g(g) + self.A_x(x)))

        # Phase (modulation)
        theta = torch.tanh(self.P_g(g) + self.P_x(x))

        # Quantum-inspired transformation
        Q = A * torch.cos(theta)

        # Project back
        Q = self.out(Q)

        return x * torch.sigmoid(Q)


# ================================
# DOWN
# ================================
class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.down = nn.Conv3d(in_ch, out_ch, kernel_size=2, stride=2)
        self.block = VNetBlock(out_ch, out_ch)

    def forward(self, x):
        return self.block(self.down(x))


# ================================
# UP (WITH QUANTUM GATE)
# ================================
class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, out_ch, 2, stride=2)

        # quantum gate instead of attention
        self.qgate = QuantumGate3D(out_ch, out_ch, out_ch // 2)

        self.block = VNetBlock(out_ch * 2, out_ch)

    def forward(self, x1, x2):
        x1 = self.up(x1)

        x2 = x2[:, :, :x1.shape[2], :x1.shape[3], :x1.shape[4]]

        # apply quantum gating
        x2 = self.qgate(x1, x2)

        x = torch.cat([x2, x1], dim=1)
        return self.block(x)


# ================================
# FULL MODEL
# ================================
class QuantumVNet3D(nn.Module):
    def __init__(self):
        super().__init__()

        # lightweight (Kaggle safe)
        self.in_block = VNetBlock(1, 16)

        self.down1 = Down(16, 32)
        self.down2 = Down(32, 64)
        self.down3 = Down(64, 128)

        self.up1 = Up(128, 64)
        self.up2 = Up(64, 32)
        self.up3 = Up(32, 16)

        self.out = nn.Conv3d(16, NUM_CLASSES, 1)

    def forward(self, x):
        x1 = self.in_block(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)

        x = self.up1(x4, x3)
        x = self.up2(x, x2)
        x = self.up3(x, x1)

        return self.out(x)   # logits


# ================================
# INIT
# ================================
model = QuantumVNet3D().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)


### Execution: Training & Visual Comparison
For this architecture, we run two separate training pipelines to compare loss functions:
* **V1:** Optimized using Focal Tversky + Dice Loss.
* **V2:** Optimized using Cross-Entropy + Dice Loss.

During training, we dynamically track our comprehensive metrics and save the best model checkpoint based on the validation **Dice Score**. Finally, we visualize the middle Z-slice to compare the Ground Truth mask against the V1 and V2 predictions.

In [ ]:
# ── V1 SETUP ──
loss_function_v1 = FocalTverskyDiceLoss(NUM_CLASSES)
model_v1 = QuantumVNet3D(NUM_CLASSES)  
if torch.cuda.device_count() > 1:
    model_v1 = nn.DataParallel(model_v1)
model_v1 = model_v1.to(DEVICE)
optimizer_v1 = torch.optim.AdamW(model_v1.parameters(), lr=LR)
scaler_v1 = torch.cuda.amp.GradScaler()

best_score_v1 = -1e9
train_losses_v1 = []

# Dynamically track all new metrics
metric_keys = ["dice", "iou", "precision", "recall", "f1", "specificity", "hd95", "cldice", "betti"]
val_history_v1 = {k: [] for k in metric_keys}

for epoch in range(NUM_EPOCHS):
    # TRAIN
    model_v1.train()
    running_loss = 0
    for img, lbl in train_loader:
        img = img.to(DEVICE)
        lbl = lbl.to(DEVICE)
        optimizer_v1.zero_grad()
        with torch.cuda.amp.autocast():
            pred = model_v1(img)
            loss = loss_function_v1(pred, lbl)
        scaler_v1.scale(loss).backward()
        scaler_v1.step(optimizer_v1)
        scaler_v1.update()
        running_loss += loss.item()
    avg_train_loss = running_loss / len(train_loader)
    train_losses_v1.append(avg_train_loss)

    # VALIDATION
    model_v1.eval()
    batch_metrics = {k: [] for k in metric_keys}
    
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            logits = model_v1(x)
            probs = torch.softmax(logits, dim=1)
            _, pred = torch.max(probs, dim=1)
            
            # Convert to numpy for the new metric function
            pred_np = pred.cpu().numpy()
            y_np = y.cpu().numpy()
            
            m = compute_metrics(pred_np, y_np, NUM_CLASSES)
            
            for k in metric_keys:
                batch_metrics[k].append(m[k])

    # Calculate average for the epoch
    epoch_metrics = {k: np.mean(v) for k, v in batch_metrics.items()}
    
    # Store in history
    for k in metric_keys:
        val_history_v1[k].append(epoch_metrics[k])

    # Use average Dice as the primary score for saving the best model
    score = epoch_metrics["dice"]

    print(
        f"[V1] Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Loss: {avg_train_loss:.4f} | "
        f"Dice: {epoch_metrics['dice']:.4f} | "
        f"IoU: {epoch_metrics['iou']:.4f} | "
        f"F1: {epoch_metrics['f1']:.4f} | "
        f"HD95: {epoch_metrics['hd95']:.4f} | "
        f"Score: {score:.4f}"
    )

    if score > best_score_v1:
        print(f"[V1] Saving improved model ({best_score_v1:.4f} -> {score:.4f})")
        checkpoint_v1 = {
            "epoch": epoch + 1,
            "model_state": model_v1.module.state_dict() if isinstance(model_v1, nn.DataParallel) else model_v1.state_dict(),
            "optimizer_state": optimizer_v1.state_dict(),
            "best_score": score,
            "train_losses": train_losses_v1,
            "val_history": val_history_v1 # Saved the whole dictionary of metrics
        }
        torch.save(checkpoint_v1, "/kaggle/working/best_model_quantumvnet_v1.pth")
        best_score_v1 = score

In [ ]:
# ── V2 SETUP ──
loss_function_v2 = CrossEntropyDiceLoss(NUM_CLASSES)
model_v2 = QuantumVNet3D(NUM_CLASSES) 
if torch.cuda.device_count() > 1:
    model_v2 = nn.DataParallel(model_v2)
model_v2 = model_v2.to(DEVICE)
optimizer_v2 = torch.optim.AdamW(model_v2.parameters(), lr=LR)
scaler_v2 = torch.cuda.amp.GradScaler()

best_score_v2 = -1e9
train_losses_v2 = []

# Dynamically track all new metrics
metric_keys = ["dice", "iou", "precision", "recall", "f1", "specificity", "hd95", "cldice", "betti"]
val_history_v2 = {k: [] for k in metric_keys}

for epoch in range(NUM_EPOCHS):
    # TRAIN
    model_v2.train()
    running_loss = 0
    for img, lbl in train_loader:
        img = img.to(DEVICE)
        lbl = lbl.to(DEVICE)
        optimizer_v2.zero_grad()
        with torch.cuda.amp.autocast():
            pred = model_v2(img)
            loss = loss_function_v2(pred, lbl)
        scaler_v2.scale(loss).backward()
        scaler_v2.step(optimizer_v2)
        scaler_v2.update()
        running_loss += loss.item()
    avg_train_loss = running_loss / len(train_loader)
    train_losses_v2.append(avg_train_loss)

    # VALIDATION
    model_v2.eval()
    batch_metrics = {k: [] for k in metric_keys}
    
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            logits = model_v2(x)
            probs = torch.softmax(logits, dim=1)
            _, pred = torch.max(probs, dim=1)
            
            # Convert to numpy for the new metric function
            pred_np = pred.cpu().numpy()
            y_np = y.cpu().numpy()
            
            m = compute_metrics(pred_np, y_np, NUM_CLASSES)
            
            for k in metric_keys:
                batch_metrics[k].append(m[k])

    # Calculate average for the epoch
    epoch_metrics = {k: np.mean(v) for k, v in batch_metrics.items()}
    
    # Store in history
    for k in metric_keys:
        val_history_v2[k].append(epoch_metrics[k])

    # Use average Dice as the primary score for saving the best model
    score = epoch_metrics["dice"]

    print(
        f"[V2] Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Loss: {avg_train_loss:.4f} | "
        f"Dice: {epoch_metrics['dice']:.4f} | "
        f"IoU: {epoch_metrics['iou']:.4f} | "
        f"F1: {epoch_metrics['f1']:.4f} | "
        f"HD95: {epoch_metrics['hd95']:.4f} | "
        f"Score: {score:.4f}"
    )

    if score > best_score_v2:
        print(f"[V2] Saving improved model ({best_score_v2:.4f} -> {score:.4f})")
        checkpoint_v2 = {
            "epoch": epoch + 1,
            "model_state": model_v2.module.state_dict() if isinstance(model_v2, nn.DataParallel) else model_v2.state_dict(),
            "optimizer_state": optimizer_v2.state_dict(),
            "best_score": score,
            "train_losses": train_losses_v2,
            "val_history": val_history_v2 # Saved the whole dictionary of metrics
        }
        torch.save(checkpoint_v2, "/kaggle/working/best_model_quantumvnet_v2.pth")
        best_score_v2 = score

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math
import torch

# ── LOAD CHECKPOINTS ──
# Update these paths to match the specific model trained
CKPT_PATH_V1 = "/kaggle/working/best_model_quantumvnet_v1.pth"
CKPT_PATH_V2 = "/kaggle/working/best_model_quantumvnet_v2.pth"

ckpt_v1 = torch.load(CKPT_PATH_V1, map_location=DEVICE, weights_only=False)
ckpt_v2 = torch.load(CKPT_PATH_V2, map_location=DEVICE, weights_only=False)

# ── EXTRACT BEST EPOCH INDICES ──
best_idx_v1 = ckpt_v1["epoch"] - 1
best_idx_v2 = ckpt_v2["epoch"] - 1

# ── PRINT ALL BEST METRICS ──
print("="*50)
print(f" BEST METRICS: V1 (FocalTversky+Dice) [Epoch {ckpt_v1['epoch']}]")
print("="*50)
print(f"  • Score (Primary): {ckpt_v1['best_score']:.4f}")
for k, v in ckpt_v1["val_history"].items():
    print(f"  • {k.capitalize():<12}: {v[best_idx_v1]:.6f}")

print("\n" + "="*50)
print(f" BEST METRICS: V2 (CrossEntropy+Dice) [Epoch {ckpt_v2['epoch']}]")
print("="*50)
print(f"  • Score (Primary): {ckpt_v2['best_score']:.4f}")
for k, v in ckpt_v2["val_history"].items():
    print(f"  • {k.capitalize():<12}: {v[best_idx_v2]:.6f}")
print("="*50)

# ── DYNAMIC METRIC PLOTS ──
metric_keys = list(ckpt_v1["val_history"].keys())
num_plots = len(metric_keys) + 1 # +1 for training loss
cols = 4
rows = math.ceil(num_plots / cols)

plt.figure(figsize=(cols * 4.5, rows * 4))

# 1. Plot Train Loss
plt.subplot(rows, cols, 1)
plt.plot(ckpt_v1["train_losses"], label="V1 FocalTversky", color='blue')
plt.plot(ckpt_v2["train_losses"], label="V2 CE+Dice", color='orange')
plt.title("Train Loss", fontweight='bold')
plt.xlabel("Epochs")
plt.legend()
plt.grid(True, alpha=0.3)

# 2. Plot All Validation Metrics Dynamically
for i, key in enumerate(metric_keys):
    plt.subplot(rows, cols, i + 2)
    plt.plot(ckpt_v1["val_history"][key], label="V1", color='blue')
    plt.plot(ckpt_v2["val_history"][key], label="V2", color='orange')
    
    # Mark the best epoch point
    plt.scatter(best_idx_v1, ckpt_v1["val_history"][key][best_idx_v1], color='blue', s=50, zorder=5)
    plt.scatter(best_idx_v2, ckpt_v2["val_history"][key][best_idx_v2], color='red', s=50, zorder=5)
    
    plt.title(f"Val {key.capitalize()}", fontweight='bold')
    plt.xlabel("Epochs")
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── LOAD BOTH MODELS ──.
model_v1 = QuantumVNet3D().to(DEVICE)
model_v1.load_state_dict(ckpt_v1["model_state"])
model_v1.eval()

model_v2 = QuantumVNet3D().to(DEVICE)
model_v2.load_state_dict(ckpt_v2["model_state"])
model_v2.eval()

# ── GET A VALIDATION SAMPLE ──
img, lbl = next(iter(val_loader))
img = img.to(DEVICE)

with torch.no_grad():
    pred_v1 = torch.argmax(model_v1(img), dim=1)
    pred_v2 = torch.argmax(model_v2(img), dim=1)

slice_idx = img.shape[-1] // 2

# ── PREDICTION PLOTS ──
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(img[0, 0, :, :, slice_idx].cpu(), cmap='gray')
axes[0].set_title("Input (CT/MR)", fontweight='bold')

axes[1].imshow(lbl[0, :, :, slice_idx].cpu(), cmap='tab20')
axes[1].set_title("Ground Truth", fontweight='bold')

axes[2].imshow(pred_v1[0, :, :, slice_idx].cpu(), cmap='tab20')
axes[2].set_title("V1: FocalTversky + Dice", fontweight='bold')

axes[3].imshow(pred_v2[0, :, :, slice_idx].cpu(), cmap='tab20')
axes[3].set_title("V2: CrossEntropy + Dice", fontweight='bold')

for ax in axes:
    ax.axis("off")

plt.suptitle(
    f"Visual Comparison — Middle Slice ({slice_idx})\n"
    f"V1 Best Dice: {ckpt_v1['val_history']['dice'][best_idx_v1]:.4f} | "
    f"V2 Best Dice: {ckpt_v2['val_history']['dice'][best_idx_v2]:.4f}", 
    fontsize=14, fontweight='bold', y=1.05
)
plt.tight_layout()
plt.show()